# Probabilidad de las hazañas del fútbol
**Universidad del Istmo · Análisis de Datos**  
**Catedrático:** Juan Andrés García Porres

Didvin Nohel Estrada Pineda · 14092  
Jose Humberto Najar Venavente · 13661  
Pablo Rodolfo Alexander Flores Mollinedo · 14643

## Guía de ejecución
Este es un solo notebook con dos partes completas y claramente separadas:
1. **Leverkusen:** historia de la Bundesliga, invicto, simulación y extra de rescates, con sus seis gráficas.
2. **Cabo Verde:** eliminatoria, escenarios del grupo y ruta al título, con sus tres gráficas.

Sube este archivo a [Google Colab](https://colab.research.google.com/) y selecciona **Ejecutar todas**. No requiere GPU, credenciales ni subir un ZIP. Los datos necesarios están incorporados como una proyección compacta y se verifican con SHA-256. Se conservan todos los goles, autogoles y el último evento del segundo tiempo de cada partido, así como los registros históricos y Elo utilizados. Los originales completos y sus huellas permanecen en el paquete del proyecto. La instalación de bibliotecas necesita conexión; los cálculos y la carga de datos no consultan a los proveedores.

Cada parte mantiene su propio directorio de resultados y puede ejecutarse desde su encabezado después de la instalación. Al final se descarga un único ZIP con ambas carpetas. Las fuentes se filtran antes de incorporarlas; se libera la memoria al pasar de Leverkusen a Cabo Verde. El muestreo sigue usando 10 000 repeticiones y las mismas semillas.


In [ ]:
%pip -q install numpy pandas scipy matplotlib


# PARTE I · LEVERKUSEN Y EXTRA

En este bloque se cargan los datos y se ejecutan exclusivamente los cálculos y las gráficas de esta parte.

In [ ]:
from pathlib import Path
import json, math
from IPython.display import display, Image, Markdown
import gc
# Evita que IPython conserve resultados grandes en su caché Out.
try:
    get_ipython().cache_size = 0
    get_ipython().displayhook.cache_size = 0
except NameError:
    pass
ROOT = Path.cwd() / "leverkusen_colab"
RAW = ROOT / "data/raw"
OUT = ROOT / "outputs"
PROC = ROOT / "data/processed"
for directory in (RAW, PROC, OUT / "figures"):
    directory.mkdir(parents=True, exist_ok=True)


## Datos verificables
Se restaura exclusivamente la copia de datos correspondiente a este caso. La procedencia se conserva en `data/sources.json`.

In [ ]:
# Copia histórica incorporada: no se consulta ningún servidor de datos.
# Extrae solo los nombres del manifiesto y verifica cada archivo antes de analizarlo.
import base64, io, zipfile, hashlib, gc
MANIFEST = {'germany.csv': {'url': 'https://raw.githubusercontent.com/jalapic/engsoccerdata/master/data-raw/germany.csv', 'sha256': 'd546c0c035367c2a8dfd717be64e6cb3783c6e2904d7bd77e3937ceabbd021b9', 'original_sha256': 'e955aef5dabb62909d584ae3ba77c994078d031976b52241aebff190bf67d931', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'clubelo_archive.csv': {'url': 'https://raw.githubusercontent.com/xgabora/Club-Football-Match-Data/main/data/EloRatings.csv', 'sha256': '0231a89498e6ded70bdc6d7eb2b11a97983645aab3b13e700fc3dff077707c95', 'original_sha256': '1bbd3f8b171407483cf6e0a5a37fa983b9e9bf53a62a92f537bbe40d7dbb36f1', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'leverkusen_matches_statsbomb.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/9/281.json', 'sha256': 'c225b2715c777a4de0f0a85f56fe461986d1a3597dd64dbec90e692d4e2ff241', 'original_sha256': '13dff90f126d9f73da410ae3292b2773744657d3ba968a2b7a9c02135033581c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895320.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895320.json', 'sha256': '1fd11fe99f43400f53adc7aa0daa98da6a324e52d79e7ed7b3457c39521c4765', 'original_sha256': '87b2347238c7946d625b1ba2a20042aef2f906dafbd7a65c7624871b13ae2bce', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895292.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895292.json', 'sha256': '79fe330e46563d5b9c6d9a028f925f928e130bcb5b22bd2ad6899fe8ddfccd5d', 'original_sha256': '546f6f0d4f71b40f81ef82c45b4a19f652ca9152943f45f934b9faca8a5cb728', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895158.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895158.json', 'sha256': '5e5eb61910fb3dd78c66579198afc24a90d409c37e7130547bcd62c0f7b6833c', 'original_sha256': '0c5cd7dd15224d4faf9202997ce3839821478804d32dc7cea10b51b8badf54bc', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895340.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895340.json', 'sha256': '8414c03ae41ae367162dbb7a9077290b003c0e1bb9effe91c04b61582225251d', 'original_sha256': '0d4780e3f067882d0bc01477fd71e817f109e14112b61a48df1739a0e8eb8faa', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895107.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895107.json', 'sha256': '68577163104a58ed32b39c3613ee9645b8e35332f18214d02becc8848758cea5', 'original_sha256': '26f6dab51a13ad9877daae29da74bab0be7c5813743994c26e7f4de14719b58c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895286.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895286.json', 'sha256': '1177b33388d72d36d85b528f9a52a836cac9ced8bf99179015ae5b23e4de09d4', 'original_sha256': 'd2a8eba6cb9b41f012fd9546754e97d783cc18229cf1b793e0fae23979f388e7', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895302.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895302.json', 'sha256': 'fe491a4f478b07e9fd81e26efd12ad2b7835f5abc88486b7731157eadd83bd39', 'original_sha256': '47743ba159a4c89679faeee9cce1156312807f838311d453bea18f82e3a3a9d5', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895333.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895333.json', 'sha256': 'a21c6fcba763078907ae09fc1943fa6014bd42c6f5288f36f77858fdb8704ddf', 'original_sha256': '46aa0d7156256796a7c81f9661035d126c0ec0deda43783d0fb36bd906b60c15', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895348.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895348.json', 'sha256': '6abd3a395de71ca33bc720ede6373968fe52aeb96ce790f291aeedc164a42eaf', 'original_sha256': '380a71237d8be1baa5a1dc3ba8bb9c8966e5234bc8d3757f74c384bf9ff3345d', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895250.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895250.json', 'sha256': '9e1827ccf43c190e73c3780e6f898a1e34fe73449d4529f5b53c1930acda0c9d', 'original_sha256': '1a5c6a7614efda5c8431fc32cca5936a7ee552f315b44300780a2159a0406306', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895220.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895220.json', 'sha256': 'f7c2c71a8d991eff7e453123865f1a1b10846ce48b7eb82a62aa02e12e13caf9', 'original_sha256': '3b09b6522dcff98c44ea55163ac977b513c177356b9bddb46dd30a3981bfef64', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895266.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895266.json', 'sha256': 'd41103a293ec9ae88f53cecb53a59804d68dc73dae47f39193014f6311b84d2c', 'original_sha256': 'b404b1b3a14f786a2b34c1f5f1d2bbd90b095c2740f6e47d2950e00c6eb5e1b8', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895275.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895275.json', 'sha256': 'fbdf835f99bf5c6bffb62f360ebe4ff1ffc78f1fc01a4c4a9d1c935a1526f354', 'original_sha256': '53f11b132bde1ab1c7a1e63ceb9d4cd790e9a44bc65209e664c8ab3ad2b5b5d8', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895180.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895180.json', 'sha256': '0ecc62bd0cf0a1cba598eab3d22a721fb5abd9cca91d6d152f34a71192ecf87d', 'original_sha256': 'aeddc426aa8b7af3e7afdefab252e67819e045199bd95b18a2fa944e2a85999e', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895134.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895134.json', 'sha256': 'd0986ee7107e181a6f0a47e64bf5e3487dd2a98b63dd0342f355d48ac02cbc65', 'original_sha256': 'b1e71963d53ad18cdaa5f7c8133d8fe3882246716f7cad090db81e7812532327', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895121.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895121.json', 'sha256': '62563afc2a0c6097461767c38c2c9458aa5d95d00b9f594414a83810f5605187', 'original_sha256': 'de2b29ac79817c66c0089166a8078eeff234e18c72b1627b8d9eed3d0a4e587c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895074.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895074.json', 'sha256': 'e7ff6f914abd02c875860c65d085012163728209ecafc8c1024ee5151da6139a', 'original_sha256': 'b43f2c87a77eeca368247ac290fa85297183c14f1a1eb7f5d9b335173205427d', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895139.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895139.json', 'sha256': 'caf9a75a407ac7c1406f0c82926684407f5aba5588a0f93c63474aa6efd789e2', 'original_sha256': '6f5e7165f5f8ff0b5619f81ed6695ac07e19093f68608a70da70c8976e114650', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895086.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895086.json', 'sha256': 'f8b82a247b039ce7f5eba92d37a5650eb75da4c0218aafed0466b3664bec4ec3', 'original_sha256': '84d8ce36e84efbe1b168d39efe0b166c50a48da3cf587986a6455cbb06d6e119', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895258.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895258.json', 'sha256': '8b1157f06bf77db09dd4e55f237977b6333a92acc87ee95698dac921693236e9', 'original_sha256': '3510b91fa0173e72bfe95d4f93af8e9534e9712ec53a553dcde4031ab1ef89fb', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895309.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895309.json', 'sha256': '72c53e614742506db6aa7792eac8ff992fd744219d8a34131077bc4b9aa3900c', 'original_sha256': '20e5d981bcb0352348ba8b4236dd1ff43dad316de290e59b8207278e4cd00466', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895244.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895244.json', 'sha256': '011b61a8d7177bb1fc6f144a021f735987b25454eb08eb2e28fc6f18521777f1', 'original_sha256': 'b5415201274952e62d48231e2f4983673009cb6b62009c08604f0e490ae10040', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895232.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895232.json', 'sha256': '0eddb0ff30436c6d711e259dd37f6a91ae6c5f54a2145eb22f84d28a7f1e4228', 'original_sha256': '71c47769f161da2a728cf2cefc32fdfc8ba9db81ac1e054488aaaae46f389f48', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895202.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895202.json', 'sha256': '36eda89a047bb1d042b3fed2455411c471e8ae2ba2b9203418f3104292462c35', 'original_sha256': '39fa9916efaaf66a639fe4872986d7192043ac974ab4816f998affc0a3db8e0b', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895194.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895194.json', 'sha256': 'c912f39da9c6192fc7bebffaf810d6cac54475ecb4a2f80736e50c49965e9e0d', 'original_sha256': '3404cacac2ec167e75064dd3855f8da236bdda068c0d900cdfc0fb36cc5ad188', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895182.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895182.json', 'sha256': '3bfaad60f7fc24a472390abd2e3468bf6ef4996204687ac09556ce3a98eace9e', 'original_sha256': 'b3f3f8a620647bdd66fdab4f57ea60492aedef5156eaa08af800be1f976c952b', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895210.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895210.json', 'sha256': '99db6b6bce4e672adfbf2f385d966adda259b429d47bee46f86ca1e1c0cc2a66', 'original_sha256': '30e14ba83b6674b778d8d13b2307db5355181a5fd25cb15c5eb1b2f840827444', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895113.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895113.json', 'sha256': '46a487fcf329d3ad59ed78e336e063fcb92257c700843f39ea1c6d07a8657064', 'original_sha256': '09b45ef8b6773cd3f6b6e4d4bdd7346e18c8225654a0a1a48ee7577062b5aad7', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895095.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895095.json', 'sha256': '7004721dd2f52fe849a543506c69475a7d95115275dcd8f5652648b360255e61', 'original_sha256': 'cf37c272322ea2daa3f102783d60dd637585eb029dda0e94e052ab9f5c648a29', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895167.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895167.json', 'sha256': 'a69f220513faaf865bb1ccf5deb874fb08a2314c79946b80e35176c0cb50061b', 'original_sha256': 'a3bfe843f8a95147dd9fdb8dcfb41bc3b940fa84a037aa5c5ef6340f52e87e6c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895153.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895153.json', 'sha256': '73978dc740aa676634578ac933e1dbebfbb7ae773952870390cbcc1e959479c7', 'original_sha256': 'ff08bf0f3c1d9717eb60a64571ba9aeb413f61475dd0d14529e39cb00cc47b6b', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895067.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895067.json', 'sha256': 'e40913196284a96ce0926f3fac5a8a69e7cdbe5c0af469c08e07bf869f9e25b2', 'original_sha256': '8bba4bec9d870aa5b88eab122ba50f33c89d038580893e330e9c0ec41e9a946a', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895060.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895060.json', 'sha256': '71649f2fe619480a8114e5a6594586bf9360a82e5e63e7a2d382423de4acb9ca', 'original_sha256': '1b2fcf16823fbf04f4ae40492d492304f8d54fdb991cebc7fdcaa2e1d72e5585', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'events/3895052.json': {'url': 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3895052.json', 'sha256': '99f1390e0de6d894a99826d2723b75745800899a63483bf68702d258998ae9ce', 'original_sha256': 'ecc00b4e7ddd0154532329091d5d82fd7de4b9f6aa8da174788816001e40e605', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}}
DATA_ARCHIVE_BASE64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(DATA_ARCHIVE_BASE64))) as archive:
    for name, meta in MANIFEST.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != meta["sha256"]:
            raise ValueError("Los datos incorporados no coinciden: " + name)
        dest = RAW / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
(ROOT / "data/sources.json").write_text(json.dumps(MANIFEST, indent=2))
print(f"{len(MANIFEST)} archivos restaurados y verificados; sin descarga HTTP.")
del DATA_ARCHIVE_BASE64, data, archive
gc.collect()


## Notación, variables y supuestos

### Qué es Elo y qué representa

Elo es un sistema de valoración de fuerza relativa, llamado así por Arpad Elo; no es una sigla. Asigna una puntuación a cada equipo. La diferencia entre dos valoraciones permite calcular el resultado esperado: un rival con más Elo representa, según el sistema, una mayor dificultad.

En este trabajo, E representa P(victoria) + ½ P(empate), no la probabilidad de ganar. Los Elo observados son entradas fijas durante cada simulación. Aunque algunos proveedores los redondean a enteros, se tratan como una escala numérica de intervalo: importan las diferencias, no afirmar que un equipo sea el doble de fuerte por tener el doble de Elo.

### Discreta, continua o categórica: por qué importa

Una variable discreta toma valores aislados y contables, como 0, 1 o 2 goles. Una continua representa una magnitud que puede variar en un intervalo, como el instante de un gol. Equipo, rival y condición local o visitante son variables categóricas: sus etiquetas no son cantidades que debamos promediar.

Para variables discretas calculamos probabilidades mediante sumas de masas, por ejemplo la probabilidad de cero derrotas. Para una variable realmente continua, la probabilidad de un valor puntual es cero bajo un modelo con densidad; se integran intervalos. Los tiempos aquí se registran con resolución finita y no se ajusta una densidad de tiempos.

### Leverkusen: variables discretas del análisis

Los goles a favor y en contra por partido son conteos enteros no negativos; se modelan con Poisson. Su diferencia es un entero, positivo, cero o negativo, y se modela con Skellam. El resultado es una categoría ordenada —derrota, empate o victoria— obtenida al comparar esos goles.

Las derrotas de Leverkusen en 34 jornadas toman valores de 0 a 34. El indicador de invicto toma 0 o 1; el total de invictos entre N temporadas es un conteo de 0 a N. Para campeones históricos, las derrotas tienen un límite real dado por los partidos jugados; Poisson es una aproximación con soporte no acotado, no una identidad del torneo.

### Leverkusen: tiempos, parámetros y estimadores

El instante del gol y el tiempo restante se miden en minutos y segundos: conceptualmente son magnitudes continuas, observadas sobre una rejilla temporal. El déficit de goles y el número de rescates son discretos. Equipo y temporada identifican observaciones; no suponemos que temporadas de distintas épocas tengan idénticas condiciones.

Las tasas Poisson, la ventaja local, los Elo y las probabilidades p y q son parámetros o entradas numéricas fijadas en el cálculo; no reciben una distribución aleatoria propia. La proporción simulada K/N sí es un estimador discreto, con incrementos de 1/N. Que se escriba con decimales no la convierte en continua; por eso usamos un intervalo binomial exacto.

### Enlace probabilístico
$\Delta=R_i-R_j+h$, $E=(1+10^{-\Delta/400})^{-1}$. Ajustamos las tasas de dos Poisson independientes para que $E=P(G_i>G_j)+\frac12P(G_i=G_j)$. La diferencia de goles sigue Skellam. El parámetro $\tau$ es la escala base de goles, no una suma fija de tasas para todos los rivales.

$N$ es el número de simulaciones, $K$ el número de éxitos y $\hat p=K/N$. Los intervalos exactos binomiales miden solamente precisión Monte Carlo. Las fuerzas quedan fijas; no modelamos incertidumbre de Elo.

In [ ]:
"""Modelos de marcadores y clasificación usados en ambos estudios."""

from functools import lru_cache
import numpy as np
from scipy.optimize import brentq
from scipy.stats import skellam, beta


@lru_cache(maxsize=4096)
def goal_rates(delta: float, total: float = 2.7):
    """Invierte el Elo esperado E=P(G)+P(E)/2 para dos Poisson independientes."""
    expected = 1 / (1 + 10 ** (-delta / 400))

    def residual(t):
        a = (total / 2) * np.exp(t)
        b = (total / 2) * np.exp(-t)
        return skellam.sf(0, a, b) + 0.5 * skellam.pmf(0, a, b) - expected

    t = brentq(residual, -8, 8)
    return (total / 2) * np.exp(t), (total / 2) * np.exp(-t)


def probabilities(delta, total=2.7):
    a, b = goal_rates(float(delta), float(total))
    return np.array([skellam.sf(0, a, b), skellam.pmf(0, a, b), skellam.cdf(-1, a, b)])


def interval(k, n, alpha=0.05):
    if not n:
        return [None, None]
    return [
        0.0 if k == 0 else float(beta.ppf(alpha / 2, k, n - k + 1)),
        1.0 if k == n else float(beta.ppf(1 - alpha / 2, k + 1, n - k)),
    ]


def summarize(mask):
    k = int(np.sum(mask))
    n = len(mask)
    return {"successes": k, "n": n, "p": k / n if n else None, "ci95": interval(k, n)}


def rank_table(scores, head_to_head=True, rng=None):
    """Puntos; minitabla H2H (Mundial); DG; GF; sorteo residual explícito.

    scores[i,j] son los goles de i contra j, sumados en ida y vuelta.
    match_points se entrega aparte para no confundir agregado con resultados.
    """
    goals, points = scores
    n = len(goals)
    gf = goals.sum(axis=1)
    gd = gf - goals.sum(axis=0)
    pts = points.sum(axis=1)
    tie = np.zeros(n) if rng is None else rng.random(n)
    keys = []
    for i in range(n):
        tied = np.flatnonzero(pts == pts[i])
        hpts = points[i, tied].sum()
        hgd = goals[i, tied].sum() - goals[tied, i].sum()
        hgf = goals[i, tied].sum()
        key = (
            (pts[i], hpts, hgd, hgf, gd[i], gf[i], tie[i])
            if head_to_head
            else (pts[i], gd[i], gf[i], hpts, hgd, hgf, tie[i])
        )
        keys.append(key)
    return sorted(range(n), key=lambda i: keys[i], reverse=True), pts, gd, gf


def group_sim(
    teams, ratings, n, rng, double=False, forced_cv=False, total=2.7, home_adv=0
):
    """Simula marcadores y devuelve clasificación y resultados sin usar datos futuros."""
    m = len(teams)
    g = np.zeros((n, m, m), dtype=np.int16)
    pt = np.zeros_like(g)
    fixtures = []
    draw_prob = 1.0
    for i in range(m):
        for j in range(i + 1, m):
            for home, away in [(i, j), (j, i)] if double else [(i, j)]:
                delta = ratings[teams[home]] - ratings[teams[away]] + home_adv
                a, b = goal_rates(float(delta), float(total))
                x = rng.poisson(a, n)
                y = rng.poisson(b, n)
                if forced_cv and "CV" in [teams[home], teams[away]]:
                    # Marcador condicionado a empate: P(k,k) proporcional a Poisson(a,k)Poisson(b,k).
                    from scipy.stats import poisson

                    ks = np.arange(25)
                    w = poisson.pmf(ks, a) * poisson.pmf(ks, b)
                    draw_prob *= w.sum()
                    w /= w.sum()
                    x = rng.choice(ks, size=n, p=w)
                    y = x.copy()
                g[:, home, away] += x
                g[:, away, home] += y
                pt[:, home, away] += 3 * (x > y) + (x == y)
                pt[:, away, home] += 3 * (y > x) + (x == y)
                fixtures.append((teams[home], teams[away], x, y))
    order = np.zeros((n, m), dtype=int)
    stats = np.zeros((n, m, 3), dtype=int)
    for k in range(n):
        o, p, d, f = rank_table((g[k], pt[k]), head_to_head=not double, rng=rng)
        order[k] = o
        stats[k] = np.stack([p, d, f], axis=1)
    return {
        "teams": teams,
        "order": order,
        "stats": stats,
        "fixtures": fixtures,
        "p_three_draws": draw_prob,
    }


In [ ]:
from pathlib import Path
import json, math
import numpy as np, pandas as pd
from scipy.stats import poisson, nbinom, skellam



RAW = ROOT / "data/raw"
OUT = ROOT / "outputs"
PROC = ROOT / "data/processed"
for p in [OUT, PROC, OUT / "figures"]:
    p.mkdir(exist_ok=True, parents=True)
SEED = 14643
N = 10000
ALIAS = {
    "Bayern Munchen": "Bayern Munich",
    "FC Bayern Munchen": "Bayern Munich",
    "Bayer Leverkusen": "Leverkusen",
    "RasenBallsport Leipzig": "RB Leipzig",
    "Borussia Dortmund": "Dortmund",
    "Eintracht Frankfurt": "Ein Frankfurt",
    "Frankfurter SG Eintracht": "Ein Frankfurt",
    "1. FC Koln": "FC Koln",
    "Bor. Monchengladbach": "MGladbach",
    "Borussia Monchengladbach": "MGladbach",
    "1. FSV Mainz 05": "Mainz",
    "1. FC Union Berlin": "Union Berlin",
    "VfB Stuttgart": "Stuttgart",
    "SC Freiburg": "Freiburg",
    "FC Augsburg": "Augsburg",
    "VfL Bochum": "Bochum",
    "VfL Wolfsburg": "Wolfsburg",
    "SV Darmstadt 98": "Darmstadt",
    "1. FC Heidenheim": "Heidenheim",
    "1899 Hoffenheim": "Hoffenheim",
}




## Reconstrucción histórica
Se calcula la tabla de cada temporada. La corrección de Stuttgart 1991/92 está documentada en el código porque el archivo de partidos es incompleto.

In [ ]:
def historical():
    df = pd.read_csv(RAW / "germany.csv")
    df = df[(df.tier == 1) & df.Season.between(1963, 2023)].copy()
    df["home"] = df.home.replace(ALIAS)
    df["visitor"] = df.visitor.replace(ALIAS)
    champions = []
    for season, ms in df.groupby("Season"):
        clubs = sorted(set(ms.home) | set(ms.visitor))
        rows = []
        for club in clubs:
            h = ms[ms.home == club]
            a = ms[ms.visitor == club]
            w = int((h.hgoal > h.vgoal).sum() + (a.vgoal > a.hgoal).sum())
            d = int((h.hgoal == h.vgoal).sum() + (a.vgoal == a.hgoal).sum())
            l = len(h) + len(a) - w - d
            gf = int(h.hgoal.sum() + a.vgoal.sum())
            ga = int(h.vgoal.sum() + a.hgoal.sum())
            rows.append(
                dict(
                    team=club,
                    played=len(h) + len(a),
                    wins=w,
                    draws=d,
                    losses=l,
                    gf=gf,
                    ga=ga,
                    points=(2 if season < 1995 else 3) * w + d,
                    gd=gf - ga,
                )
            )
        champ = max(
            rows,
            key=lambda x: (
                x["points"],
                x["gf"] / max(1, x["ga"]) if season < 1969 else x["gd"],
                x["gf"],
            ),
        )
        if season == 1991:
            # La fuente de partidos conserva 340 de los 380 cruces de esta campaña.
            # Registro completo contrastado con DSFS y Sportschau (38 jornadas).
            assert len(ms) == 340
            champ = dict(
                team="Stuttgart",
                played=38,
                wins=21,
                draws=10,
                losses=7,
                gf=62,
                ga=32,
                points=52,
                gd=30,
            )
        else:
            assert len(ms) == (240 if season in [1963, 1964] else 306), (
                season,
                len(ms),
            )
        champions.append({"end_year": int(season + 1), **champ})
    ch = pd.DataFrame(champions)
    ch[ch.end_year <= 2023].to_csv(PROC / "champions_1964_2023.csv", index=False)
    hist = ch[ch.end_year <= 2023]
    recent = hist[hist.end_year >= 2005]
    assert len(hist) == 60 and (hist.team == "Bayern Munich").sum() == 32
    assert len(recent) == 19 and (recent.team == "Bayern Munich").sum() == 15
    fits = []
    for name, sub in [
        ("Todos", hist),
        ("Bayern", hist[hist.team == "Bayern Munich"]),
        ("Otros", hist[hist.team != "Bayern Munich"]),
    ]:
        mu = float(sub.losses.mean())
        var = float(sub.losses.var(ddof=1))
        rate = sub.losses.sum() / sub.played.sum()
        row = {
            "group": name,
            "n": len(sub),
            "mean": mu,
            "variance": var,
            "p_zero": math.exp(-mu),
            "p_zero_34": math.exp(-34 * rate),
            "empirical_zero": float((sub.losses == 0).mean()),
        }
        if var > mu:
            r = mu * mu / (var - mu)
            p = r / (r + mu)
            row.update(nb_r=r, nb_p_zero=float(nbinom.pmf(0, r, p)))
        fits.append(row)
    return (
        df,
        ch,
        {"non_bayern_all": 28 / 60, "non_bayern_recent": 4 / 19, "fits": fits},
    )

In [ ]:
df, ch, history = historical()
display(pd.read_csv(PROC / "champions_1964_2023.csv"))
display(pd.DataFrame(history["fits"]))

## Invicto, Monte Carlo y extra de rescates
Se reconstruyen las probabilidades por partido, 10 000 temporadas y los cuatro rescates. Q es un diagnóstico retrospectivo, no un efecto causal de la suerte.

In [ ]:
def leverkusen(df, ch):
    # Filtra por bloques: no conserva todo el archivo Elo en memoria.
    elo = pd.concat(
        [chunk[(chunk.country == "GER") & (chunk.date == "2023-08-15")]
         for chunk in pd.read_csv(RAW / "clubelo_archive.csv", chunksize=20000)],
        ignore_index=True,
    )
    actual = df[df.Season == 2023]
    clubs = sorted(actual.home.unique())
    cut = "2023-08-15"
    er = (
        elo[(elo.country == "GER") & (elo.date == cut) & elo.club.isin(clubs)]
        .copy()
        .sort_values("elo", ascending=False)
    )
    assert len(er) == 18
    er.to_csv(PROC / "bundesliga_elo_2023_08_15.csv", index=False)
    ratings = er.set_index("club").elo.to_dict()
    train = df[df.Season.between(2018, 2022)]
    total = float((train.hgoal + train.vgoal).mean())
    hfa = 60.0
    fixtures = actual[
        (actual.home == "Leverkusen") | (actual.visitor == "Leverkusen")
    ].sort_values("Date")
    rows = []
    for m in fixtures.itertuples():
        home = m.home == "Leverkusen"
        opp = m.visitor if home else m.home
        delta = ratings["Leverkusen"] - ratings[opp] + (hfa if home else -hfa)
        pw, pd_, pl = probabilities(delta, total)
        rows.append(
            {
                "date": m.Date,
                "opponent": opp,
                "home": home,
                "elo_delta": delta,
                "p_win": pw,
                "p_draw": pd_,
                "p_loss": pl,
                "p_unbeaten": pw + pd_,
            }
        )
    f = pd.DataFrame(rows)
    f.to_csv(PROC / "leverkusen_probabilities.csv", index=False)
    pb = float(f[f.opponent == "Bayern Munich"].p_unbeaten.prod())
    po = float(f[f.opponent != "Bayern Munich"].p_unbeaten.prod())
    p = pb * po
    rng = np.random.default_rng(SEED)
    u = rng.random((N, 34))
    loss = (u > f.p_unbeaten.to_numpy()).sum(axis=1)
    win = (u < f.p_win.to_numpy()).sum(axis=1)
    draw = 34 - win - loss
    pd.DataFrame(
        {
            "season": np.arange(1, N + 1),
            "wins": win,
            "draws": draw,
            "losses": loss,
            "points": 3 * win + draw,
        }
    ).to_csv(OUT / "leverkusen_10000.csv", index=False)
    # Reconstrucción cronológica de goles de los 34 partidos, incluidos autogoles.
    matches = json.loads((RAW / "leverkusen_matches_statsbomb.json").read_text())
    goals = []
    audit = []
    rescues = []
    for m in sorted(matches, key=lambda z: z["match_date"]):
        hn = m["home_team"]["home_team_name"]
        an = m["away_team"]["away_team_name"]
        levhome = hn == "Bayer Leverkusen"
        ours = 0
        theirs = 0
        at85 = None
        late = []
        state_start = 85.0
        qstate = None
        events = json.loads((RAW / f"events/{m['match_id']}.json").read_text())
        for e in events:
            typ = e["type"]["name"]
            goal = e.get("shot", {}).get("outcome", {}).get("name") == "Goal"
            if not goal and typ != "Own Goal Against":
                continue
            islev = e["team"]["name"] == "Bayer Leverkusen"
            islev = (not islev) if typ == "Own Goal Against" else islev
            minute = e["minute"] + e["second"] / 60
            # El agregado de minutos de StatsBomb reinicia segunda parte en 45.
            if e["period"] == 2 and minute >= 85 and at85 is None:
                at85 = (ours, theirs)
            before = (ours, theirs)
            if e["period"] == 2 and minute >= 85 and qstate is None:
                if ours < theirs:
                    qstate = (ours, theirs)
                elif not islev and theirs + 1 > ours:
                    qstate = (ours, theirs + 1)
                    state_start = minute
            if islev:
                ours += 1
            else:
                theirs += 1
            goals.append(
                {
                    "match_id": m["match_id"],
                    "date": m["match_date"],
                    "opponent": an if levhome else hn,
                    "period": e["period"],
                    "minute": e["minute"],
                    "second": e["second"],
                    "leverkusen_goal": islev,
                    "player": e.get("player", {}).get("name", "Autogol"),
                    "score_lev": ours,
                    "score_opp": theirs,
                }
            )
            if (
                e["period"] == 2
                and minute >= 85
                and islev
                and before[0] < before[1]
                and ours >= theirs
            ):
                late.append(goals[-1])
        if at85 is None:
            at85 = (ours, theirs)
        expected = (
            (m["home_score"], m["away_score"])
            if levhome
            else (m["away_score"], m["home_score"])
        )
        assert (ours, theirs) == expected, (m["match_id"], ours, theirs, expected)
        audit.append(
            {
                "date": m["match_date"],
                "opponent": an if levhome else hn,
                "score85_lev": at85[0],
                "score85_opp": at85[1],
                "final_lev": ours,
                "final_opp": theirs,
                "rescued": bool(late),
            }
        )
        if late:
            l = late[0]
            rate_ours = float(train.hgoal.mean() if levhome else train.vgoal.mean())
            rate_opp = float(train.vgoal.mean() if levhome else train.hgoal.mean())
            # Tasas históricas liga antes de 2023/24, estado al 85 y 5 min de añadido supuestos.
            end_second = max(
                e["minute"] + e["second"] / 60 for e in events if e["period"] == 2
            )
            deficit = qstate[1] - qstate[0]
            remaining = end_second - state_start
            q = float(
                skellam.sf(
                    deficit - 1, rate_ours * remaining / 90, rate_opp * remaining / 90
                )
            )
            rescues.append(
                {
                    "date": m["match_date"],
                    "opponent": an if levhome else hn,
                    "at85": f"{at85[0]}-{at85[1]}",
                    "minute": f"{l['minute']}:{l['second']:02d}",
                    "player": l["player"],
                    "final": f"{ours}-{theirs}",
                    "q": q,
                    "home": levhome,
                    "deficit": deficit,
                    "state_minute": state_start,
                    "state_score": f"{qstate[0]}-{qstate[1]}",
                    "remaining": remaining,
                    "final_whistle": end_second,
                }
            )
    pd.DataFrame(goals).to_csv(PROC / "leverkusen_goals.csv", index=False)
    pd.DataFrame(audit).to_csv(PROC / "leverkusen_85min_audit.csv", index=False)
    pd.DataFrame(rescues).to_csv(PROC / "rescues.csv", index=False)
    Q = float(np.prod([r["q"] for r in rescues]))
    sensitivity = []
    for extra in [0, 3, 5, 8]:
        qs = []
        for r in rescues:
            a = train.hgoal.mean() if r["home"] else train.vgoal.mean()
            b = train.vgoal.mean() if r["home"] else train.hgoal.mean()
            qs.append(
                skellam.sf(
                    r["deficit"] - 1,
                    a * max(0.1, 90 + extra - r["state_minute"]) / 90,
                    b * max(0.1, 90 + extra - r["state_minute"]) / 90,
                )
            )
        sensitivity.append(
            {
                "added_minutes": extra,
                "Q": float(np.prod(qs)),
                "adjusted": p * float(np.prod(qs)),
            }
        )
    elo_sens = []
    for h in [0, 60, 100]:
        for boost in [0, 100, 200]:
            vals = [
                probabilities(
                    ratings["Leverkusen"]
                    + boost
                    - ratings[r.opponent]
                    + (h if r.home else -h),
                    total,
                )[:2].sum()
                for r in f.itertuples()
            ]
            elo_sens.append(
                {"hfa": h, "leverkusen_elo_boost": boost, "p": float(np.prod(vals))}
            )
    return {
        "elo_date": cut,
        "total_goals": total,
        "hfa": hfa,
        "p_bayern2": pb,
        "p_other32": po,
        "p_unbeaten": p,
        "simulation": summarize(loss == 0),
        "expected_unbeaten_10000": N * p,
        "mean_losses": float(loss.mean()),
        "rescues": rescues,
        "Q": Q,
        "inverse_Q": 1 / Q,
        "adjusted": p * Q,
        "late_sensitivity": sensitivity,
        "elo_sensitivity": elo_sens,
    }

In [ ]:
leverkusen_result = leverkusen(df, ch)
r = {"seed": SEED, "n": N, "history": history, "leverkusen": leverkusen_result}
print(json.dumps(r, indent=2, ensure_ascii=False))
display(pd.DataFrame(leverkusen_result["rescues"]))

## Gráficas de esta parte
El código siguiente dibuja todas las figuras del caso y conserva versiones PNG y SVG.

In [ ]:
def charts_case(df, ch, r):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 11,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.dpi": 150,
            "savefig.bbox": "tight",
        }
    )
    red = "#B62835"
    blue = "#176B9A"
    gray = "#7B858D"

    def save(name):
        plt.tight_layout()
        plt.savefig(OUT / f"figures/{name}.png", dpi=200)
        plt.savefig(OUT / f"figures/{name}.svg")
        plt.close()

    h = ch[ch.end_year <= 2023]
    counts = h.team.value_counts()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(
        counts.index[::-1],
        counts.values[::-1],
        color=[red if x == "Bayern Munich" else gray for x in counts.index[::-1]],
    )
    ax.set_xlabel("Títulos entre 1964 y 2023")
    save("champions")
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.scatter(
        h.end_year,
        np.zeros(len(h)),
        c=[red if x == "Bayern Munich" else blue for x in h.team],
        s=80,
    )
    ax.set_yticks([])
    ax.set_xlabel("Año de finalización de temporada")
    ax.set_title("Rojo: Bayern Múnich    Azul: otros campeones")
    save("timeline")
    f = pd.read_csv(PROC / "bundesliga_elo_2023_08_15.csv")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(
        f.club[::-1],
        f.elo[::-1],
        color=[red if x == "Leverkusen" else gray for x in f.club[::-1]],
    )
    ax.set_xlim(1400, 2000)
    ax.set_xlabel("Elo al 15 de agosto de 2023")
    save("elo")
    sim = pd.read_csv(OUT / "leverkusen_10000.csv")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.hist(
        sim.losses,
        bins=np.arange(-0.5, sim.losses.max() + 1.5),
        color=red,
        edgecolor="white",
    )
    ax.set_xlabel("Derrotas en 34 partidos")
    ax.set_ylabel("Temporadas simuladas")
    ax.set_title(f"10 000 temporadas · {int((sim.losses == 0).sum())} invictas")
    save("montecarlo")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(14)
    for fit, c in zip(r["history"]["fits"], [gray, red, blue]):
        ax.plot(x, poisson.pmf(x, fit["mean"]), marker="o", label=fit["group"], color=c)
    ax.set_xlabel("Derrotas del campeón")
    ax.set_ylabel("Probabilidad Poisson")
    ax.legend()
    save("poisson")
    fig, ax = plt.subplots(figsize=(9, 4))
    rr = r["leverkusen"]["rescues"]
    ax.barh([x["opponent"] for x in rr], [100 * x["q"] for x in rr], color=red)
    ax.set_xlabel("Probabilidad de rescate desde el minuto 85 (%)")
    save("rescues")


In [ ]:
(OUT / "results.json").write_text(json.dumps(r, indent=2, ensure_ascii=False))
charts_case(df, ch, r)
for figure in sorted((OUT / "figures").glob("*.png")):
    display(Markdown("### " + figure.stem.replace("_", " ")))
    display(Image(filename=str(figure),width=800))

## Verificación numérica
Se contrasta el resultado calculado con el del informe. Si cambias la semilla, N o el modelo, no se espera la misma realización Monte Carlo.

In [ ]:
REFERENCE = {'seed': 14643, 'n': 10000, 'history': {'non_bayern_all': 0.4666666666666667, 'non_bayern_recent': 0.21052631578947367, 'fits': [{'group': 'Todos', 'n': 60, 'mean': 4.55, 'variance': 2.8957627118644065, 'p_zero': 0.010567204383852655, 'p_zero_34': 0.010473164070506559, 'empirical_zero': 0.0}, {'group': 'Bayern', 'n': 32, 'mean': 4.125, 'variance': 3.0161290322580645, 'p_zero': 0.016163494588165874, 'p_zero_34': 0.016163494588165874, 'empirical_zero': 0.0}, {'group': 'Otros', 'n': 28, 'mean': 5.035714285714286, 'variance': 2.406084656084656, 'p_zero': 0.006501552491753531, 'p_zero_34': 0.006364866474243006, 'empirical_zero': 0.0}]}, 'leverkusen': {'elo_date': '2023-08-15', 'total_goals': 3.142483660130719, 'hfa': 60.0, 'p_bayern2': 0.11738494655663884, 'p_other32': 7.89191227773592e-06, 'p_unbeaten': 9.263917009517128e-07, 'simulation': {'successes': 0, 'n': 10000, 'p': 0.0, 'ci95': [0.0, 0.0003688199146187622]}, 'expected_unbeaten_10000': 0.009263917009517128, 'mean_losses': 10.7175, 'rescues': [{'date': '2023-09-15', 'opponent': 'Bayern Munich', 'at85': '1-1', 'minute': '93:46', 'player': 'Exequiel Alejandro Palacios', 'final': '2-2', 'q': 0.14551132695960833, 'home': False, 'deficit': 1, 'state_minute': 85.35, 'state_score': '1-2', 'remaining': 13.050000000000011, 'final_whistle': 98.4}, {'date': '2024-03-30', 'opponent': 'Hoffenheim', 'at85': '0-1', 'minute': '87:24', 'player': 'Robert Andrich', 'final': '2-1', 'q': 0.16727818921568072, 'home': True, 'deficit': 1, 'state_minute': 85.0, 'state_score': '0-1', 'remaining': 11.183333333333337, 'final_whistle': 96.18333333333334}, {'date': '2024-04-21', 'opponent': 'Borussia Dortmund', 'at85': '0-1', 'minute': '96:52', 'player': 'Josip Stanišić', 'final': '1-1', 'q': 0.15123013489280768, 'home': False, 'deficit': 1, 'state_minute': 85.0, 'state_score': '0-1', 'remaining': 13.799999999999997, 'final_whistle': 98.8}, {'date': '2024-04-27', 'opponent': 'VfB Stuttgart', 'at85': '1-2', 'minute': '95:59', 'player': 'Robert Andrich', 'final': '2-2', 'q': 0.18423259682607374, 'home': True, 'deficit': 1, 'state_minute': 85.0, 'state_score': '1-2', 'remaining': 12.733333333333334, 'final_whistle': 97.73333333333333}], 'Q': 0.0006781736835333622, 'inverse_Q': 1474.548517998938, 'adjusted': 6.2825447222916e-10, 'late_sensitivity': [{'added_minutes': 0, 'Q': 3.1959966103303785e-05, 'adjusted': 2.960744736079868e-11}, {'added_minutes': 3, 'Q': 0.00016117889679081066, 'adjusted': 1.4931479235555966e-10}, {'added_minutes': 5, 'Q': 0.000329333964477986, 'adjusted': 3.050922515339324e-10}, {'added_minutes': 8, 'Q': 0.000722736961359359, 'adjusted': 6.695375229743689e-10}], 'elo_sensitivity': [{'hfa': 0, 'leverkusen_elo_boost': 0, 'p': 1.4212311156965513e-06}, {'hfa': 0, 'leverkusen_elo_boost': 100, 'p': 0.0003033769451685277}, {'hfa': 0, 'leverkusen_elo_boost': 200, 'p': 0.009589799965793228}, {'hfa': 60, 'leverkusen_elo_boost': 0, 'p': 9.263917009517128e-07}, {'hfa': 60, 'leverkusen_elo_boost': 100, 'p': 0.0002149547331302565}, {'hfa': 60, 'leverkusen_elo_boost': 200, 'p': 0.007494974585882285}, {'hfa': 100, 'leverkusen_elo_boost': 0, 'p': 4.359069972573508e-07}, {'hfa': 100, 'leverkusen_elo_boost': 100, 'p': 0.00011674468769366359}, {'hfa': 100, 'leverkusen_elo_boost': 200, 'p': 0.004827130057376752}]}}
def compare(a, b, path="resultado"):
    if isinstance(a, dict):
        assert a.keys() == b.keys(), path
        for key in a: compare(a[key], b[key], path + "." + key)
    elif isinstance(a, list):
        assert len(a) == len(b), path
        for i, (x,y) in enumerate(zip(a,b)): compare(x,y,path+f"[{i}]")
    elif isinstance(a, (int,float)):
        assert np.isclose(a,b,rtol=1e-9,atol=1e-12), (path,a,b)
    else: assert a == b, (path,a,b)


compare(r, REFERENCE)
assert len(list((OUT / "figures").glob("*.png"))) == 6
print("Resultados de esta parte verificados.")

## Fórmulas con datos, despejes y resultados

### Historia: sustituir los conteos

$\widehat p_{1964:2023}=\frac{28}{60}=0.466667=46.6667\%$

$\widehat p_{2005:2023}=\frac{4}{19}=0.210526=21.0526\%$

Se cuentan temporadas cuyo campeón no fue Bayern. La diferencia es 21.0526 − 46.6667 = −25.6140 puntos porcentuales. Describe las ventanas históricas; no es la probabilidad de invicto de Leverkusen.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
print("Historia:",28/60,4/19,"diferencia pp:",100*(4/19-28/60))

### Ejemplo: Leverkusen recibe a Leipzig

$\Delta=1748.23-1825.38+60=-17.15$

$E=\frac{1}{1+10^{17.15/400}}=0.475339$

Se usan las valoraciones del 15 de agosto de 2023 y 60 puntos Elo de localía. E es el resultado esperado —victoria más medio empate—, por lo que todavía falta convertirlo en probabilidades de marcador.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
delta=1748.23-1825.38+60
print("Delta y expectativa:",delta,1/(1+10**(-delta/400)))

### Leipzig: tasas y probabilidad de no perder

$\tau=3.142484,\quad\lambda_L=1.514991,\quad\lambda_R=1.629582$

$p_1=P(G_L\geq G_R)=0.357246+0.236187=0.593433$

Las tasas se obtienen resolviendo numéricamente la ecuación Elo–Poisson. Al sumar victoria y empate se obtiene la probabilidad de superar este partido sin derrota. Los cálculos internos conservan la precisión completa; las cifras mostradas se redondean.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
a,b=goal_rates(-17.15,r["leverkusen"]["total_goals"])
v,d,lost=probabilities(-17.15,r["leverkusen"]["total_goals"])
print("Tasas:",a,b,"victoria, empate, derrota:",v,d,lost,"no perder:",v+d)

### Temporada completa: producto con datos

$P(A)=(0.117384947)(7.891912\times10^{-6})$

$P(A)=9.263917\times10^{-7}\quad\Rightarrow\quad1/P(A)=1079457$

El primer factor corresponde a los dos partidos contra Bayern; el segundo, al producto de los otros 32. El inverso equivale aproximadamente a una temporada invicta por 1.08 millones de temporadas bajo este modelo, sin implicar un tiempo real de espera histórico.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
p=r["leverkusen"]["p_bayern2"]*r["leverkusen"]["p_other32"]
print("Producto e inverso:",p,1/p)

### Monte Carlo: cero éxitos e incertidumbre

$E[K]=10000(9.263917\times10^{-7})=0.00926392$

$\widehat p=\frac{0}{10000}=0,\quad U=1-(0.025)^{1/10000}=0.000368820$

Para el intervalo binomial exacto bilateral del 95 %, el límite inferior es cero y el superior equivale a 0.036882 %. La probabilidad de observar cero invictos es (1 − P(A)) elevado a 10 000, aproximadamente 99.08 %. Por eso la simulación es compatible con el producto analítico.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
p=r["leverkusen"]["p_unbeaten"]
print("Esperados, P(cero), IC95:",10000*p,(1-p)**10000,interval(0,10000))

### Poisson: medias históricas sustituidas

$\widehat\lambda=\frac{273}{60}=4.55,\quad P(D=0)=e^{-4.55}=0.0105672$

$e^{-132/32}=0.0161635,\quad e^{-141/28}=0.00650155$

Los 60 campeones suman 273 derrotas: Bayern aporta 132 en 32 títulos y los demás 141 en 28. Los resultados equivalen a 1.05672 %, 1.61635 % y 0.65016 %. Son probabilidades ajustadas para campeones históricos, no para un club cualquiera antes de comenzar la liga.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
print("Poisson: todos, Bayern y otros:",np.exp(-273/60),np.exp(-132/32),np.exp(-141/28))

### Rescate ante Bayern: tiempo y tasas

$T=98.40-85.35=13.05,\quad\mu_L=1.393464\frac{13.05}{90}=0.202052$

$\mu_B=1.749020\frac{13.05}{90}=0.253608,\quad q_1=P(X-Y\geq1)=0.145511$

Leverkusen pasa a perder en el minuto 85.35; el final observado fue 98.40. Se escalan las medias históricas de goles visitantes y locales a los 13.05 minutos restantes. La cola Skellam permite goles de ambos equipos y mide la probabilidad de compensar el déficit de uno.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
from scipy.stats import skellam
mu_l=np.float64(1.3934640522875816)*13.05/90
mu_b=np.float64(1.7490196078431373)*13.05/90
print("Tasas restantes y rescate:",mu_l,mu_b,skellam.sf(0,mu_l,mu_b))

### Cuatro rescates: producto y ajuste

$Q=0.145511\times0.167278\times0.151230\times0.184233=6.781737\times10^{-4}$

$P(A)Q=(9.263917\times10^{-7} )(6.781737\times10^{-4} )=6.282545\times10^{-10}$

El orden es Bayern, Hoffenheim, Dortmund y Stuttgart. El inverso de Q es aproximadamente 1474.55. El último producto es el índice heurístico pedido en el ejercicio; no debe interpretarse como una probabilidad causal del invicto sin suerte.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
q=np.array([x["q"] for x in r["leverkusen"]["rescues"]])
print("q, Q, inverso y ajuste:",q,np.prod(q),1/np.prod(q),r["leverkusen"]["p_unbeaten"]*np.prod(q))

### Despeje de Elo: de E a la diferencia

$E^{-1}-1=10^{-\Delta/400}\ \Rightarrow\ \Delta=400\log_{10}\!\left(\frac{E}{1-E}\right)$

$\Delta=400\log_{10}\!\left(\frac{0.475339192}{1-0.475339192}\right)\approx-17.15$

Se parte de la ecuación logística, se invierte E y se resta uno. Después se aplica logaritmo base diez y se multiplica por −400. Este despeje recupera la diferencia de fuerza, incluida la localía; no convierte E en probabilidad de victoria.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
E=1/(1+10**(17.15/400))
print("Delta recuperada:",400*np.log10(E/(1-E)))

### Tasas Poisson: solución numérica del enlace

$f(t)=P(X>Y)+\frac12P(X=Y)-E=0$

$t=-0.036456989,\quad\lambda_L=\frac{3.142484}{2}e^t=1.514991,\quad\lambda_R=\frac{3.142484}{2}e^{-t}=1.629582$

Para cada valor de t se evalúan las probabilidades Skellam con tasas (τ/2)exp(t) y (τ/2)exp(−t). Brent busca una raíz en el intervalo [−8, 8]. La solución iguala el resultado esperado del marcador con el Elo; se comprueba el residuo en lugar de presentar un supuesto despeje algebraico.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
from scipy.stats import skellam
a,b=goal_rates(-17.15,r["leverkusen"]["total_goals"])
E=1/(1+10**(17.15/400))
print("t y residuo:",np.log(a/(r["leverkusen"]["total_goals"]/2)),skellam.sf(0,a,b)+.5*skellam.pmf(0,a,b)-E)

### Derivación del estimador Poisson

$\ell(\lambda)=-n\lambda+\left(\sum_s D_s\right)\log\lambda-\sum_s\log(D_s!)$

$\ell^{\prime}(\lambda)=-n+\frac{\sum_sD_s}{\lambda}=0\ \Rightarrow\ \widehat\lambda=\frac{273}{60}=4.55$

Se multiplica la masa Poisson de las derrotas de cada campeón y se toma logaritmo. Al derivar e igualar a cero se obtiene la media. La segunda derivada es −273/λ², negativa para λ positivo, por lo que el punto es un máximo. Sustituir cero derrotas en la masa da exp(−4.55).

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
lam=273/60
print("Media, segunda derivada y P(cero):",lam,-273/lam**2,np.exp(-lam))

### Despeje del límite con cero éxitos

$P(K=0\mid p=U)=(1-U)^{10000}=0.025$

$1-U=(0.025)^{1/10000}\ \Rightarrow\ U=1-(0.025)^{1/10000}=0.000368820$

Para el intervalo exacto bilateral del 95 % se asigna 0.025 a esta cola. Se toma la raíz de orden 10 000 y se despeja U. El límite inferior es cero; multiplicar U por cien lo expresa como 0.036882 %. El valor 0.025 corresponde a un intervalo bilateral, no al límite unilateral del 95 %.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
U=1-.025**(1/10000)
print("U, porcentaje y cola recuperada:",U,100*U,(1-U)**10000)

## Conclusiones

### Conclusión estadística: Leverkusen

Se concluye que el invicto de Leverkusen fue un evento extremadamente poco probable bajo el modelo de Elo de pretemporada, goles Poisson e independencia entre partidos: su probabilidad estimada es 9.264 × 10⁻⁷. Los cero invictos observados en 10 000 simulaciones son compatibles con esa estimación y no demuestran imposibilidad. El 1.06 % obtenido con Poisson histórico responde a otra pregunta, porque considera equipos que ya fueron campeones; por ello no contradice el cálculo inicial. Los cuatro rescates identificados muestran que el logro dependió de goles tardíos decisivos, pero el índice Q no permite atribuir causalmente ese resultado a la suerte. Estadísticamente, la evidencia permite calificar el invicto como excepcional dentro de los modelos utilizados, sin identificar su probabilidad verdadera ni demostrar que las valoraciones iniciales reflejaran toda la capacidad del equipo durante la temporada.

## Referencias
- [Curley: engsoccerdata](https://github.com/jalapic/engsoccerdata)
- [Archivo ClubElo](https://github.com/xgabora/Club-Football-Match-Data)
- [StatsBomb Open Data](https://github.com/statsbomb/open-data)
- [DSFS: tablas históricas](https://www.dsfs.de/wp-content/uploads/2024/01/Bundesliga_Geschichte_Tabellen.pdf)
- [SciPy: Skellam](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skellam.html)
- Goldsman y Goldsman, libro del curso, capítulos 1 y 4–6.

Las URL documentan la procedencia; no se necesita acceder a ellas para ejecutar el análisis.

## Descargar tablas, datos y gráficas
El ZIP incluye los datos necesarios de esta parte, los procesados y los resultados.

In [ ]:
bundle = ROOT / "resultados_leverkusen.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for directory in (ROOT / "data", OUT):
        for file in directory.rglob("*"):
            if file.is_file(): z.write(file, file.relative_to(ROOT))
print("Resultados de esta parte:", bundle)


In [ ]:
# Los resultados ya están en disco; liberar objetos de Leverkusen antes de Cabo Verde.
import gc
import matplotlib.pyplot as plt
plt.close('all')
for variable in ['df','ch','history','leverkusen_result','r','REFERENCE','MANIFEST']:
    globals().pop(variable, None)
goal_rates.cache_clear()
gc.collect()


# PARTE II · CABO VERDE

En este bloque se cargan los datos y se ejecutan exclusivamente los cálculos y las gráficas de esta parte.

In [ ]:
from pathlib import Path
import json, math
from IPython.display import display, Image, Markdown
import gc
# Evita que IPython conserve resultados grandes en su caché Out.
try:
    get_ipython().cache_size = 0
    get_ipython().displayhook.cache_size = 0
except NameError:
    pass
ROOT = Path.cwd() / "cabo_verde_colab"
RAW = ROOT / "data/raw"
OUT = ROOT / "outputs"
PROC = ROOT / "data/processed"
for directory in (RAW, PROC, OUT / "figures"):
    directory.mkdir(parents=True, exist_ok=True)


## Datos verificables
Se restaura exclusivamente la copia de datos correspondiente a este caso. La procedencia se conserva en `data/sources.json`.

In [ ]:
# Copia histórica incorporada: no se consulta ningún servidor de datos.
# Extrae solo los nombres del manifiesto y verifica cada archivo antes de analizarlo.
import base64, io, zipfile, hashlib, gc
MANIFEST = {'elo_teams.tsv': {'url': 'https://www.eloratings.net/en.teams.tsv', 'sha256': 'ff4b8c9c7bcdad56e81a007726c1a655c6dabdf77bca151b1d619f24125dbc90', 'original_sha256': 'ff4b8c9c7bcdad56e81a007726c1a655c6dabdf77bca151b1d619f24125dbc90', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/AO.tsv': {'url': 'https://www.eloratings.net/Angola.tsv', 'sha256': 'b131db6fac4702e2ffc01fb28264aad0bad3bebfade10d9b29da916123a60dc3', 'original_sha256': 'b131db6fac4702e2ffc01fb28264aad0bad3bebfade10d9b29da916123a60dc3', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/AT.tsv': {'url': 'https://www.eloratings.net/Austria.tsv', 'sha256': 'b8ee1c6a7f07008b95bc1b3a1f7f0ffc31bd1cf62e27361af2cfa5e9560c5f9c', 'original_sha256': 'b8ee1c6a7f07008b95bc1b3a1f7f0ffc31bd1cf62e27361af2cfa5e9560c5f9c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/AR.tsv': {'url': 'https://www.eloratings.net/Argentina.tsv', 'sha256': 'd496632c453a2baec109a57b3cf2cf13e50d005aae5a6f3b3b7bafe0c47c7f6d', 'original_sha256': 'd496632c453a2baec109a57b3cf2cf13e50d005aae5a6f3b3b7bafe0c47c7f6d', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/AU.tsv': {'url': 'https://www.eloratings.net/Australia.tsv', 'sha256': '33ec687da46d1432cd4c049abec31caf6d18c7bb6573802b48d05f495f0b20d1', 'original_sha256': '33ec687da46d1432cd4c049abec31caf6d18c7bb6573802b48d05f495f0b20d1', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/BA.tsv': {'url': 'https://www.eloratings.net/Bosnia_and_Herzegovina.tsv', 'sha256': 'aeb570a466f46af4e9d96c7d9403a3de42780cc989f6dd612c8f40bd28bd3719', 'original_sha256': 'aeb570a466f46af4e9d96c7d9403a3de42780cc989f6dd612c8f40bd28bd3719', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CA.tsv': {'url': 'https://www.eloratings.net/Canada.tsv', 'sha256': 'd894f20c00be1ce33c9e6a53932dc675994f2368621f194f55de5b1f832a38b4', 'original_sha256': 'd894f20c00be1ce33c9e6a53932dc675994f2368621f194f55de5b1f832a38b4', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/BR.tsv': {'url': 'https://www.eloratings.net/Brazil.tsv', 'sha256': '8b9013ceeedc19f274c05d64205300d601a1c522fa6db940a3d81bd1748225b3', 'original_sha256': '8b9013ceeedc19f274c05d64205300d601a1c522fa6db940a3d81bd1748225b3', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CH.tsv': {'url': 'https://www.eloratings.net/Switzerland.tsv', 'sha256': 'e16914c94ebe9894fa99f9d775426d5c215c13aa73ffebe13411dedd6967ab97', 'original_sha256': 'e16914c94ebe9894fa99f9d775426d5c215c13aa73ffebe13411dedd6967ab97', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/BE.tsv': {'url': 'https://www.eloratings.net/Belgium.tsv', 'sha256': 'fbe26338690de4e16dbcb657aba8230928f23c2caad878a2cea761f9f744344c', 'original_sha256': 'fbe26338690de4e16dbcb657aba8230928f23c2caad878a2cea761f9f744344c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CD.tsv': {'url': 'https://www.eloratings.net/DR_Congo.tsv', 'sha256': '3baf1f9bf665cb4a9ee62e2d4bafade3ae89d40c0fb939421284919180b86bbb', 'original_sha256': '3baf1f9bf665cb4a9ee62e2d4bafade3ae89d40c0fb939421284919180b86bbb', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CI.tsv': {'url': 'https://www.eloratings.net/Ivory_Coast.tsv', 'sha256': '677c06fd6957e294621d06fbd1e73f602d42c41b6d9727e69288f16eb4ef945d', 'original_sha256': '677c06fd6957e294621d06fbd1e73f602d42c41b6d9727e69288f16eb4ef945d', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CO.tsv': {'url': 'https://www.eloratings.net/Colombia.tsv', 'sha256': 'cd37e58a138a6aa7896adb05371fe8aa381e2dc92690ca6c19252ded6bb8594b', 'original_sha256': 'cd37e58a138a6aa7896adb05371fe8aa381e2dc92690ca6c19252ded6bb8594b', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CV.tsv': {'url': 'https://www.eloratings.net/Cape_Verde.tsv', 'sha256': 'bbd02beb1c4e86d1ddc91790e888f0b6cd745723d89f0b925355a0491a83cfa8', 'original_sha256': 'bbd02beb1c4e86d1ddc91790e888f0b6cd745723d89f0b925355a0491a83cfa8', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CM.tsv': {'url': 'https://www.eloratings.net/Cameroon.tsv', 'sha256': 'b40e8be678ed6cfd91330c9d9abe6c188d0e5f217095858975859027d9acdc75', 'original_sha256': 'b40e8be678ed6cfd91330c9d9abe6c188d0e5f217095858975859027d9acdc75', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CW.tsv': {'url': 'https://www.eloratings.net/Curacao.tsv', 'sha256': '6f8833fb62ef33b943385ac2113375d6036bfa89b92b49b03f12454ab1e7a016', 'original_sha256': '6f8833fb62ef33b943385ac2113375d6036bfa89b92b49b03f12454ab1e7a016', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/CZ.tsv': {'url': 'https://www.eloratings.net/Czechia.tsv', 'sha256': 'a2b8e0585c7e601298c0341e8a94a0fb65e7bddbcf677d5da0109db2d303f520', 'original_sha256': 'a2b8e0585c7e601298c0341e8a94a0fb65e7bddbcf677d5da0109db2d303f520', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/DE.tsv': {'url': 'https://www.eloratings.net/Germany.tsv', 'sha256': '83f5e8c6f5f1ee8341bb6892fb1dd528ac5ae2b107c1a7746bb8092b27cb17f6', 'original_sha256': '83f5e8c6f5f1ee8341bb6892fb1dd528ac5ae2b107c1a7746bb8092b27cb17f6', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/DZ.tsv': {'url': 'https://www.eloratings.net/Algeria.tsv', 'sha256': '27a87d674135b35c2d63c259ac1e785371bab0fce2cfb0727b36421686556a13', 'original_sha256': '27a87d674135b35c2d63c259ac1e785371bab0fce2cfb0727b36421686556a13', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/EC.tsv': {'url': 'https://www.eloratings.net/Ecuador.tsv', 'sha256': '08260307167103beb5a75ec6b1ebefc21bdac2a0b8885069f78c46dcbe93313b', 'original_sha256': '08260307167103beb5a75ec6b1ebefc21bdac2a0b8885069f78c46dcbe93313b', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/EG.tsv': {'url': 'https://www.eloratings.net/Egypt.tsv', 'sha256': '2b94c0fc5f50c812a6ea3c7edb18d3ad6f4edfda2d183bd1a02149e033c2b19f', 'original_sha256': '2b94c0fc5f50c812a6ea3c7edb18d3ad6f4edfda2d183bd1a02149e033c2b19f', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/EN.tsv': {'url': 'https://www.eloratings.net/England.tsv', 'sha256': 'e273fc7446a9a7e47f006a71f7f594e52f67d5f1765c2c1747800bda77641b95', 'original_sha256': 'e273fc7446a9a7e47f006a71f7f594e52f67d5f1765c2c1747800bda77641b95', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/ES.tsv': {'url': 'https://www.eloratings.net/Spain.tsv', 'sha256': '206692bd541d66005779436532cecd892a48f0ee9ed1d7fcf2f2757a30f2c7a5', 'original_sha256': '206692bd541d66005779436532cecd892a48f0ee9ed1d7fcf2f2757a30f2c7a5', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/GH.tsv': {'url': 'https://www.eloratings.net/Ghana.tsv', 'sha256': 'b9dde86f11260ccc848aa29d27f90c80f21a1ab6efa1d68752a45f87ed99105e', 'original_sha256': 'b9dde86f11260ccc848aa29d27f90c80f21a1ab6efa1d68752a45f87ed99105e', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/FR.tsv': {'url': 'https://www.eloratings.net/France.tsv', 'sha256': '74dba257e945b7abe373359b8991b333a3039ffce4deab8c3f5a2ae6d4ea9d3a', 'original_sha256': '74dba257e945b7abe373359b8991b333a3039ffce4deab8c3f5a2ae6d4ea9d3a', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/HT.tsv': {'url': 'https://www.eloratings.net/Haiti.tsv', 'sha256': 'c35fe727c389903ed58a7d79e922fc9c77334ccf76773a22c314f8335ec5b9a0', 'original_sha256': 'c35fe727c389903ed58a7d79e922fc9c77334ccf76773a22c314f8335ec5b9a0', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/HR.tsv': {'url': 'https://www.eloratings.net/Croatia.tsv', 'sha256': 'db2c938167feede7c619ec27cdd6d13f5804388970b999ab90b1822df9e4d0fd', 'original_sha256': 'db2c938167feede7c619ec27cdd6d13f5804388970b999ab90b1822df9e4d0fd', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/IR.tsv': {'url': 'https://www.eloratings.net/Iran.tsv', 'sha256': '0fc9fe8a5d15c0e25ddbe7d0892201d5e980086fdeac4dfcc8f4ec1cb360e47c', 'original_sha256': '0fc9fe8a5d15c0e25ddbe7d0892201d5e980086fdeac4dfcc8f4ec1cb360e47c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/IQ.tsv': {'url': 'https://www.eloratings.net/Iraq.tsv', 'sha256': 'a5ac6d2487c7a5f015e85f493ec90a33da4395c4b8fe5f3317c5764ef3152d05', 'original_sha256': 'a5ac6d2487c7a5f015e85f493ec90a33da4395c4b8fe5f3317c5764ef3152d05', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/JO.tsv': {'url': 'https://www.eloratings.net/Jordan.tsv', 'sha256': 'd7b5f50c21ff991436862604e2963078feb36e07bc890cfbc7609280d2037de3', 'original_sha256': 'd7b5f50c21ff991436862604e2963078feb36e07bc890cfbc7609280d2037de3', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/JP.tsv': {'url': 'https://www.eloratings.net/Japan.tsv', 'sha256': '2b1712781b818df9a5986d72c889a5e2f19fa0ff0d94ac14d82eaee8e77fd5b5', 'original_sha256': '2b1712781b818df9a5986d72c889a5e2f19fa0ff0d94ac14d82eaee8e77fd5b5', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/LY.tsv': {'url': 'https://www.eloratings.net/Libya.tsv', 'sha256': 'e33ac871a91cbb371f94dfcf22814255c01ef0b8da4971487800847dd6a2b55c', 'original_sha256': 'e33ac871a91cbb371f94dfcf22814255c01ef0b8da4971487800847dd6a2b55c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/KR.tsv': {'url': 'https://www.eloratings.net/South_Korea.tsv', 'sha256': '187cd58a29ba8c22a20eb8ece59c840842e5679175f8002d878a71080ccf7599', 'original_sha256': '187cd58a29ba8c22a20eb8ece59c840842e5679175f8002d878a71080ccf7599', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/MA.tsv': {'url': 'https://www.eloratings.net/Morocco.tsv', 'sha256': 'df5bc1159149b98bd8af1af6ec5c7c745a20246f7d6336fbce4bb57313466cad', 'original_sha256': 'df5bc1159149b98bd8af1af6ec5c7c745a20246f7d6336fbce4bb57313466cad', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/NL.tsv': {'url': 'https://www.eloratings.net/Netherlands.tsv', 'sha256': 'cb4b420b4bf73b2443a681e75c91b71a3268c81124ac3a56ba6f1ecf59094825', 'original_sha256': 'cb4b420b4bf73b2443a681e75c91b71a3268c81124ac3a56ba6f1ecf59094825', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/MU.tsv': {'url': 'https://www.eloratings.net/Mauritius.tsv', 'sha256': '1d41e794e9e2a1844555de7c8eb27a2ba55c9be6e39ac0b806e372216193eb2c', 'original_sha256': '1d41e794e9e2a1844555de7c8eb27a2ba55c9be6e39ac0b806e372216193eb2c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/NO.tsv': {'url': 'https://www.eloratings.net/Norway.tsv', 'sha256': '3eb95ab459ab14a0d7a262e447b87c254bec94c87395e362fda58928eecdddfc', 'original_sha256': '3eb95ab459ab14a0d7a262e447b87c254bec94c87395e362fda58928eecdddfc', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/NZ.tsv': {'url': 'https://www.eloratings.net/New_Zealand.tsv', 'sha256': 'b050ef64d9a5ec7358cfd4ea2d4d871097e3a41da5409804ec31953a13ce1241', 'original_sha256': 'b050ef64d9a5ec7358cfd4ea2d4d871097e3a41da5409804ec31953a13ce1241', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/MX.tsv': {'url': 'https://www.eloratings.net/Mexico.tsv', 'sha256': '3867aad69ffd5db9042db818e1fe8321c1a6b76af299e2bde01bd44f080525f2', 'original_sha256': '3867aad69ffd5db9042db818e1fe8321c1a6b76af299e2bde01bd44f080525f2', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/PA.tsv': {'url': 'https://www.eloratings.net/Panama.tsv', 'sha256': 'e7cd1dd393c862abbfab886402e10291753626b05d1a80bdabda8e993a53bf2f', 'original_sha256': 'e7cd1dd393c862abbfab886402e10291753626b05d1a80bdabda8e993a53bf2f', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/PT.tsv': {'url': 'https://www.eloratings.net/Portugal.tsv', 'sha256': 'cb8d3475cae88e00c6f5fe7afeaae849902c060484b008a56640a6a7b460a9ce', 'original_sha256': 'cb8d3475cae88e00c6f5fe7afeaae849902c060484b008a56640a6a7b460a9ce', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/PY.tsv': {'url': 'https://www.eloratings.net/Paraguay.tsv', 'sha256': '08f3416ed34e1ed805ef6d18f8ee746d320316980e0a3154c0188155802a2dcc', 'original_sha256': '08f3416ed34e1ed805ef6d18f8ee746d320316980e0a3154c0188155802a2dcc', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/SN.tsv': {'url': 'https://www.eloratings.net/Senegal.tsv', 'sha256': 'ec659d1773c2963641b57f8ece10483ee1b2442f359430798f7da099f19dfbc3', 'original_sha256': 'ec659d1773c2963641b57f8ece10483ee1b2442f359430798f7da099f19dfbc3', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/SE.tsv': {'url': 'https://www.eloratings.net/Sweden.tsv', 'sha256': 'c526f53dc1afe78041be66d85a6591468caed11a977448269da865be6ba73742', 'original_sha256': 'c526f53dc1afe78041be66d85a6591468caed11a977448269da865be6ba73742', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/QA.tsv': {'url': 'https://www.eloratings.net/Qatar.tsv', 'sha256': '33eccef3ff59d9e424ad5ae345c8ffb8bea9333be990411dcbdc38c99fb5eb1a', 'original_sha256': '33eccef3ff59d9e424ad5ae345c8ffb8bea9333be990411dcbdc38c99fb5eb1a', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/SA.tsv': {'url': 'https://www.eloratings.net/Saudi_Arabia.tsv', 'sha256': '8c48499ce2d3d4d12f6026ad850f4edd876dd572add668a16d572f5f93e7c7d1', 'original_sha256': '8c48499ce2d3d4d12f6026ad850f4edd876dd572add668a16d572f5f93e7c7d1', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/TN.tsv': {'url': 'https://www.eloratings.net/Tunisia.tsv', 'sha256': '32854545c06ed4b713130f979852923705ea8b13be7bc1baf50b40b8e2c07036', 'original_sha256': '32854545c06ed4b713130f979852923705ea8b13be7bc1baf50b40b8e2c07036', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/US.tsv': {'url': 'https://www.eloratings.net/United_States.tsv', 'sha256': 'af9b857b997893d344b61ee098ccf1e340c69eed395671fdce42b1ff188d4641', 'original_sha256': 'af9b857b997893d344b61ee098ccf1e340c69eed395671fdce42b1ff188d4641', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/TR.tsv': {'url': 'https://www.eloratings.net/Turkey.tsv', 'sha256': 'f36788bc8174c89704415720e7524f4e397f87c4ce6cedcae3cef7c519706bfc', 'original_sha256': 'f36788bc8174c89704415720e7524f4e397f87c4ce6cedcae3cef7c519706bfc', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/SZ.tsv': {'url': 'https://www.eloratings.net/Eswatini.tsv', 'sha256': '987691c65ffdf5e5659dbefd9bf3ccce1d2336dd0d0f35d6f64af1c4f12e1d4b', 'original_sha256': '987691c65ffdf5e5659dbefd9bf3ccce1d2336dd0d0f35d6f64af1c4f12e1d4b', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/UZ.tsv': {'url': 'https://www.eloratings.net/Uzbekistan.tsv', 'sha256': '22c4c8c3fec06ce53e8e2f440c7c788b038a05fdd07b718a22784f023ba45259', 'original_sha256': '22c4c8c3fec06ce53e8e2f440c7c788b038a05fdd07b718a22784f023ba45259', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/ZA.tsv': {'url': 'https://www.eloratings.net/South_Africa.tsv', 'sha256': '5b47136c262c9b5addccb4f2b428685dc02ff31d6a48010d8d253055374b307c', 'original_sha256': '5b47136c262c9b5addccb4f2b428685dc02ff31d6a48010d8d253055374b307c', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/UY.tsv': {'url': 'https://www.eloratings.net/Uruguay.tsv', 'sha256': 'c3833721dcf2b51861eac7d369da351380aecea3480744e077647bed83b7b071', 'original_sha256': 'c3833721dcf2b51861eac7d369da351380aecea3480744e077647bed83b7b071', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}, 'national/SQ.tsv': {'url': 'https://www.eloratings.net/Scotland.tsv', 'sha256': '61ed5d7df5f4f6db2f7ec24f71105acf68c06273cde78b0775a96b01aed1caa4', 'original_sha256': '61ed5d7df5f4f6db2f7ec24f71105acf68c06273cde78b0775a96b01aed1caa4', 'representation': 'Proyección de campos y registros usados; original disponible en el proyecto'}}
DATA_ARCHIVE_BASE64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(DATA_ARCHIVE_BASE64))) as archive:
    for name, meta in MANIFEST.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != meta["sha256"]:
            raise ValueError("Los datos incorporados no coinciden: " + name)
        dest = RAW / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
(ROOT / "data/sources.json").write_text(json.dumps(MANIFEST, indent=2))
print(f"{len(MANIFEST)} archivos restaurados y verificados; sin descarga HTTP.")
del DATA_ARCHIVE_BASE64, data, archive
gc.collect()


In [ ]:
GROUPS = {'A': ['MX', 'ZA', 'KR', 'CZ'], 'B': ['CA', 'CH', 'QA', 'BA'], 'C': ['BR', 'MA', 'HT', 'SQ'], 'D': ['US', 'PY', 'AU', 'TR'], 'E': ['DE', 'CW', 'CI', 'EC'], 'F': ['NL', 'JP', 'SE', 'TN'], 'G': ['BE', 'EG', 'IR', 'NZ'], 'H': ['ES', 'CV', 'SA', 'UY'], 'I': ['FR', 'SN', 'IQ', 'NO'], 'J': ['AR', 'DZ', 'AT', 'JO'], 'K': ['PT', 'CD', 'UZ', 'CO'], 'L': ['EN', 'HR', 'GH', 'PA']}
(ROOT / "data/groups_2026.json").write_text(json.dumps(GROUPS))

## Notación, variables y supuestos

### Qué es Elo y qué representa

Elo es un sistema de valoración de fuerza relativa, llamado así por Arpad Elo; no es una sigla. Asigna una puntuación a cada equipo. La diferencia entre dos valoraciones permite calcular el resultado esperado: un rival con más Elo representa, según el sistema, una mayor dificultad.

En este trabajo, E representa P(victoria) + ½ P(empate), no la probabilidad de ganar. Los Elo observados son entradas fijas durante cada simulación. Aunque algunos proveedores los redondean a enteros, se tratan como una escala numérica de intervalo: importan las diferencias, no afirmar que un equipo sea el doble de fuerte por tener el doble de Elo.

### Discreta, continua o categórica: por qué importa

Una variable discreta toma valores aislados y contables, como 0, 1 o 2 goles. Una continua representa una magnitud que puede variar en un intervalo, como el instante de un gol. Equipo, rival y condición local o visitante son variables categóricas: sus etiquetas no son cantidades que debamos promediar.

Para variables discretas calculamos probabilidades mediante sumas de masas, por ejemplo la probabilidad de cero derrotas. Para una variable realmente continua, la probabilidad de un valor puntual es cero bajo un modelo con densidad; se integran intervalos. Los tiempos aquí se registran con resolución finita y no se ajusta una densidad de tiempos.

### Cabo Verde: goles, puntos y posiciones

Los goles de cada selección son conteos discretos Poisson. Victoria, empate y derrota son categorías ordenadas; los puntos por partido pertenecen al conjunto {0, 1, 3}. Los puntos acumulados, goles a favor y diferencia de goles son variables discretas que determinan la tabla, junto con las reglas de desempate.

La posición final es ordinal: toma valores de 1 a 6 en la eliminatoria y de 1 a 4 en el grupo mundialista. La distancia entre posiciones no mide una diferencia constante de rendimiento. País, grupo y rival son categorías nominales; la localía es una categoría que incorporamos mediante una ventaja numérica Elo.

### Cabo Verde: eventos y condicionamiento

Ganar el grupo, avanzar, obtener tres empates y ganar el título se representan con indicadores binarios. El número de clasificaciones entre N simulaciones es discreto. La condición de tres empates impone igualdad de goles en cada partido: tiene probabilidad positiva porque los marcadores son discretos.

Las probabilidades y tasas de gol son parámetros reales, no resultados continuos simulados. En escenarios condicionados, el estimador divide los éxitos dentro de la condición entre las simulaciones que la cumplen. Ese denominador puede ser aleatorio. El intervalo binomial se calcula con el número efectivamente retenido y mide únicamente precisión Monte Carlo.

### Enlace probabilístico
$\Delta=R_i-R_j+h$, $E=(1+10^{-\Delta/400})^{-1}$. Ajustamos las tasas de dos Poisson independientes para que $E=P(G_i>G_j)+\frac12P(G_i=G_j)$. La diferencia de goles sigue Skellam. El parámetro $\tau$ es la escala base de goles, no una suma fija de tasas para todos los rivales.

$N$ es el número de simulaciones, $K$ el número de éxitos y $\hat p=K/N$. Los intervalos exactos binomiales miden solamente precisión Monte Carlo. Las fuerzas quedan fijas; no modelamos incertidumbre de Elo.

In [ ]:
"""Modelos de marcadores y clasificación usados en ambos estudios."""

from functools import lru_cache
import numpy as np
from scipy.optimize import brentq
from scipy.stats import skellam, beta


@lru_cache(maxsize=4096)
def goal_rates(delta: float, total: float = 2.7):
    """Invierte el Elo esperado E=P(G)+P(E)/2 para dos Poisson independientes."""
    expected = 1 / (1 + 10 ** (-delta / 400))

    def residual(t):
        a = (total / 2) * np.exp(t)
        b = (total / 2) * np.exp(-t)
        return skellam.sf(0, a, b) + 0.5 * skellam.pmf(0, a, b) - expected

    t = brentq(residual, -8, 8)
    return (total / 2) * np.exp(t), (total / 2) * np.exp(-t)


def probabilities(delta, total=2.7):
    a, b = goal_rates(float(delta), float(total))
    return np.array([skellam.sf(0, a, b), skellam.pmf(0, a, b), skellam.cdf(-1, a, b)])


def interval(k, n, alpha=0.05):
    if not n:
        return [None, None]
    return [
        0.0 if k == 0 else float(beta.ppf(alpha / 2, k, n - k + 1)),
        1.0 if k == n else float(beta.ppf(1 - alpha / 2, k + 1, n - k)),
    ]


def summarize(mask):
    k = int(np.sum(mask))
    n = len(mask)
    return {"successes": k, "n": n, "p": k / n if n else None, "ci95": interval(k, n)}


def rank_table(scores, head_to_head=True, rng=None):
    """Puntos; minitabla H2H (Mundial); DG; GF; sorteo residual explícito.

    scores[i,j] son los goles de i contra j, sumados en ida y vuelta.
    match_points se entrega aparte para no confundir agregado con resultados.
    """
    goals, points = scores
    n = len(goals)
    gf = goals.sum(axis=1)
    gd = gf - goals.sum(axis=0)
    pts = points.sum(axis=1)
    tie = np.zeros(n) if rng is None else rng.random(n)
    keys = []
    for i in range(n):
        tied = np.flatnonzero(pts == pts[i])
        hpts = points[i, tied].sum()
        hgd = goals[i, tied].sum() - goals[tied, i].sum()
        hgf = goals[i, tied].sum()
        key = (
            (pts[i], hpts, hgd, hgf, gd[i], gf[i], tie[i])
            if head_to_head
            else (pts[i], gd[i], gf[i], hpts, hgd, hgf, tie[i])
        )
        keys.append(key)
    return sorted(range(n), key=lambda i: keys[i], reverse=True), pts, gd, gf


def group_sim(
    teams, ratings, n, rng, double=False, forced_cv=False, total=2.7, home_adv=0
):
    """Simula marcadores y devuelve clasificación y resultados sin usar datos futuros."""
    m = len(teams)
    g = np.zeros((n, m, m), dtype=np.int16)
    pt = np.zeros_like(g)
    fixtures = []
    draw_prob = 1.0
    for i in range(m):
        for j in range(i + 1, m):
            for home, away in [(i, j), (j, i)] if double else [(i, j)]:
                delta = ratings[teams[home]] - ratings[teams[away]] + home_adv
                a, b = goal_rates(float(delta), float(total))
                x = rng.poisson(a, n)
                y = rng.poisson(b, n)
                if forced_cv and "CV" in [teams[home], teams[away]]:
                    # Marcador condicionado a empate: P(k,k) proporcional a Poisson(a,k)Poisson(b,k).
                    from scipy.stats import poisson

                    ks = np.arange(25)
                    w = poisson.pmf(ks, a) * poisson.pmf(ks, b)
                    draw_prob *= w.sum()
                    w /= w.sum()
                    x = rng.choice(ks, size=n, p=w)
                    y = x.copy()
                g[:, home, away] += x
                g[:, away, home] += y
                pt[:, home, away] += 3 * (x > y) + (x == y)
                pt[:, away, home] += 3 * (y > x) + (x == y)
                fixtures.append((teams[home], teams[away], x, y))
    order = np.zeros((n, m), dtype=int)
    stats = np.zeros((n, m, 3), dtype=int)
    for k in range(n):
        o, p, d, f = rank_table((g[k], pt[k]), head_to_head=not double, rng=rng)
        order[k] = o
        stats[k] = np.stack([p, d, f], axis=1)
    return {
        "teams": teams,
        "order": order,
        "stats": stats,
        "fixtures": fixtures,
        "p_three_draws": draw_prob,
    }


In [ ]:
from pathlib import Path
import json, math
import numpy as np, pandas as pd
from scipy.stats import poisson, nbinom, skellam



RAW = ROOT / "data/raw"
OUT = ROOT / "outputs"
PROC = ROOT / "data/processed"
for p in [OUT, PROC, OUT / "figures"]:
    p.mkdir(exist_ok=True, parents=True)
SEED = 14643
N = 10000
ALIAS = {
    "Bayern Munchen": "Bayern Munich",
    "FC Bayern Munchen": "Bayern Munich",
    "Bayer Leverkusen": "Leverkusen",
    "RasenBallsport Leipzig": "RB Leipzig",
    "Borussia Dortmund": "Dortmund",
    "Eintracht Frankfurt": "Ein Frankfurt",
    "Frankfurter SG Eintracht": "Ein Frankfurt",
    "1. FC Koln": "FC Koln",
    "Bor. Monchengladbach": "MGladbach",
    "Borussia Monchengladbach": "MGladbach",
    "1. FSV Mainz 05": "Mainz",
    "1. FC Union Berlin": "Union Berlin",
    "VfB Stuttgart": "Stuttgart",
    "SC Freiburg": "Freiburg",
    "FC Augsburg": "Augsburg",
    "VfL Bochum": "Bochum",
    "VfL Wolfsburg": "Wolfsburg",
    "SV Darmstadt 98": "Darmstadt",
    "1. FC Heidenheim": "Heidenheim",
    "1899 Hoffenheim": "Hoffenheim",
}




### Selecciona el último Elo anterior al corte.

In [ ]:
def national_ratings(codes, cut):
    rows = []
    for code in codes:
        path = RAW / f"national/{code}.tsv"
        valid = []
        for line in path.read_text().splitlines():
            f = line.split("\t")
            if len(f) < 13:
                continue
            date = "-".join(f[:3])
            if date < cut and f[0].isdigit() and code in f[3:5]:
                idx = 10 if f[3] == code else 11
                try:
                    valid.append((date, int(f[idx])))
                except ValueError:
                    pass
        assert valid, code
        date, elo = valid[-1]
        rows.append({"code": code, "elo": elo, "last_match": date, "cutoff": cut})
    frame = pd.DataFrame(rows)
    frame.to_csv(PROC / f"national_elo_{cut}.csv", index=False)
    return frame.set_index("code").elo.to_dict()

### Simula todos los grupos y la clasificación de terceros; admite tres empates condicionados.

In [ ]:
def qualify_world(groups, ratings, n, rng, forced=False, total=2.7):
    # Mantiene solo los terceros y el grupo H, sin acumular doce torneos completos.
    keys = list(groups)
    third_rows = []
    h = None
    for key, teams in groups.items():
        sim = group_sim(teams, ratings, n, rng, forced_cv=forced and key == "H", total=total)
        third_rows.append(sim["stats"][np.arange(n), sim["order"][:, 2]].copy())
        if key == "H":
            h = sim
        del sim
    ci = h["teams"].index("CV")
    pos = np.argmax(h["order"] == ci, axis=1) + 1
    thirds = np.stack(third_rows, axis=1)
    del third_rows
    hi = keys.index("H")
    tie = rng.random((n, 12))
    accepted = np.zeros(n, dtype=bool)
    for j in range(n):
        rank = sorted(range(12), key=lambda k: (*thirds[j, k], tie[j, k]), reverse=True)
        accepted[j] = hi in rank[:8]
    qualified = (pos <= 2) | ((pos == 3) & accepted)
    uruguay = None
    for a, b, x, y in h["fixtures"]:
        if {a, b} == {"UY", "ES"}:
            uruguay = x >= y if a == "UY" else y >= x
    return {
        "position": pos,
        "qualified": qualified,
        "third_ok": accepted,
        "uruguay_not_lose": uruguay,
        "p_three_draws": h["p_three_draws"],
    }

### Ejecuta la eliminatoria sin repechaje, el grupo mundialista y la ruta fija del ejercicio.

In [ ]:
def cape_verde():
    groups = json.loads((ROOT / "data/groups_2026.json").read_text())
    codes = sorted(set(sum(groups.values(), [])))
    qual = ["CV", "CM", "AO", "LY", "SZ", "MU"]
    r0 = national_ratings(qual, "2023-11-15")
    r1 = national_ratings(codes, "2026-06-11")
    rng = np.random.default_rng(SEED + 1)
    q = group_sim(qual, r0, N, rng, double=True, home_adv=100)
    direct = q["order"][:, 0] == 0
    a = qualify_world(groups, r1, N, rng)
    b = qualify_world(groups, r1, N, rng, forced=True)
    rows = []
    for code in ["AR", "EG", "CH", "EN", "ES"]:
        pw, pd_, pl = probabilities(r1["CV"] - r1[code])
        rows.append(
            {
                "opponent": code,
                "elo": r1[code],
                "p_win": float(pw),
                "p_draw": float(pd_),
                "p_advance": float(pw + pd_),
            }
        )
    route = float(np.prod([x["p_advance"] for x in rows[1:]]))
    full = route * rows[0]["p_advance"]
    pd.DataFrame(rows).to_csv(PROC / "cape_verde_knockouts.csv", index=False)
    pd.DataFrame(
        {
            "trial": np.arange(1, N + 1),
            "direct_qualification": direct,
            "group_position": a["position"],
            "advance": a["qualified"],
            "three_draws_position": b["position"],
            "three_draws_advance": b["qualified"],
        }
    ).to_csv(OUT / "cape_verde_10000.csv", index=False)
    scenarios = {
        "grupo_sin_condicionar": summarize(a["qualified"]),
        "tres_empates": summarize(b["qualified"]),
        "uruguay_no_pierde_sin_condicionar_empates": summarize(
            a["qualified"][a["uruguay_not_lose"]]
        ),
        "tres_empates_y_uruguay_no_pierde": summarize(
            b["qualified"][b["uruguay_not_lose"]]
        ),
        "tres_empates_y_tercero": summarize(b["qualified"][b["position"] == 3]),
    }
    pthird = [float(np.mean(b["position"] == i)) for i in range(1, 5)]
    # Réplica independiente del recorrido eliminatorio para comparar con el producto.
    ko = (rng.random((N, 4)) < np.array([x["p_advance"] for x in rows[1:]])).all(axis=1)
    return {
        "qualifying": summarize(direct),
        "qualifying_elo": r0,
        "world_elo": r1,
        "scenarios": scenarios,
        "p_three_draws": b["p_three_draws"],
        "positions_three_draws": pthird,
        "knockouts": rows,
        "title_given_pass_argentina": route,
        "title_from_argentina": full,
        "title_from_group_fixed_route": a["qualified"].mean() * full,
        "title_from_qualifiers_fixed_route": direct.mean()
        * a["qualified"].mean()
        * full,
        "knockout_mc": summarize(ko),
        "total_goals_assumption": 2.7,
        "n": N,
    }

## Simulación completa de Cabo Verde
La ruta eliminatoria es contrafactual y supone avance en todos los empates. La condición de tres empates usa su propio denominador.

In [ ]:
cape_result = cape_verde()
r = {"seed": SEED, "n": N, "cape_verde": cape_result}
print(json.dumps(cape_result, indent=2, ensure_ascii=False))
display(pd.DataFrame(cape_result["scenarios"]).T)

## Gráficas de esta parte
El código siguiente dibuja todas las figuras del caso y conserva versiones PNG y SVG.

In [ ]:
def charts_case(r):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 11,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.dpi": 150,
            "savefig.bbox": "tight",
        }
    )
    red = "#B62835"
    blue = "#176B9A"
    gray = "#7B858D"

    def save(name):
        plt.tight_layout()
        plt.savefig(OUT / f"figures/{name}.png", dpi=200)
        plt.savefig(OUT / f"figures/{name}.svg")
        plt.close()

    cv = r["cape_verde"]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    labels = [
        "Clasificación\ndirecta",
        "Avanzar\ndel grupo",
        "Avanzar con\ntres empates",
        "Avanzar siendo\ntercero y 3 empates",
    ]
    vals = [
        cv["qualifying"]["p"],
        cv["scenarios"]["grupo_sin_condicionar"]["p"],
        cv["scenarios"]["tres_empates"]["p"],
        cv["scenarios"]["tres_empates_y_tercero"]["p"],
    ]
    ax.bar(labels, np.array(vals) * 100, color=blue)
    ax.set_ylabel("Probabilidad (%)")
    ax.set_ylim(0, 100)
    save("cape_scenarios")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ko = cv["knockouts"]
    ax.bar([x["opponent"] for x in ko], [100 * x["p_advance"] for x in ko], color=blue)
    ax.set_ylabel("Victoria o empate en 90 minutos (%)")
    ax.set_ylim(0, 100)
    save("knockouts")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(
        ["Egipto", "Suiza", "Inglaterra", "España"],
        100 * np.cumprod([x["p_advance"] for x in ko[1:]]),
        marker="o",
        color=blue,
    )
    ax.set_yscale("log")
    ax.set_ylabel("Probabilidad acumulada (%) · escala log")
    save("route")

In [ ]:
(OUT / "results.json").write_text(json.dumps(r, indent=2, ensure_ascii=False))
charts_case(r)
for figure in sorted((OUT / "figures").glob("*.png")):
    display(Markdown("### " + figure.stem.replace("_", " ")))
    display(Image(filename=str(figure),width=800))

## Verificación numérica
Se contrasta el resultado calculado con el del informe. Si cambias la semilla, N o el modelo, no se espera la misma realización Monte Carlo.

In [ ]:
REFERENCE = {'seed': 14643, 'n': 10000, 'cape_verde': {'qualifying': {'successes': 1288, 'n': 10000, 'p': 0.1288, 'ci95': [0.12229333142499799, 0.1355236733358582]}, 'qualifying_elo': {'CV': 1474, 'CM': 1625, 'AO': 1388, 'LY': 1331, 'SZ': 1286, 'MU': 1031}, 'world_elo': {'AR': 2115, 'AT': 1830, 'AU': 1777, 'BA': 1594, 'BE': 1893, 'BR': 1991, 'CA': 1788, 'CD': 1652, 'CH': 1891, 'CI': 1696, 'CO': 1982, 'CV': 1578, 'CW': 1434, 'CZ': 1740, 'DE': 1932, 'DZ': 1772, 'EC': 1938, 'EG': 1696, 'EN': 2024, 'ES': 2157, 'FR': 2064, 'GH': 1511, 'HR': 1911, 'HT': 1548, 'IQ': 1607, 'IR': 1772, 'JO': 1680, 'JP': 1906, 'KR': 1758, 'MA': 1827, 'MX': 1875, 'NL': 1947, 'NO': 1914, 'NZ': 1562, 'PA': 1730, 'PT': 1989, 'PY': 1833, 'QA': 1421, 'SA': 1576, 'SE': 1712, 'SN': 1860, 'SQ': 1782, 'TN': 1628, 'TR': 1910, 'US': 1726, 'UY': 1892, 'UZ': 1714, 'ZA': 1518}, 'scenarios': {'grupo_sin_condicionar': {'successes': 2336, 'n': 10000, 'p': 0.2336, 'ci95': [0.22533550162339555, 0.24201991459218494]}, 'tres_empates': {'successes': 9531, 'n': 10000, 'p': 0.9531, 'ci95': [0.9487722265009725, 0.9571613584926532]}, 'uruguay_no_pierde_sin_condicionar_empates': {'successes': 619, 'n': 2606, 'p': 0.2375287797390637, 'ci95': [0.22129982865995163, 0.2543494479856197]}, 'tres_empates_y_uruguay_no_pierde': {'successes': 2414, 'n': 2576, 'p': 0.937111801242236, 'ci95': [0.9270358477158427, 0.9461794111632077]}, 'tres_empates_y_tercero': {'successes': 8352, 'n': 8679, 'p': 0.9623228482544072, 'ci95': [0.9581009654247273, 0.9662305922494389]}}, 'p_three_draws': 0.0014893366159703285, 'positions_three_draws': [0.0002, 0.1177, 0.8679, 0.0142], 'knockouts': [{'opponent': 'AR', 'elo': 2115, 'p_win': 0.01750517348558367, 'p_draw': 0.05193136785161987, 'p_advance': 0.06943654133720353}, {'opponent': 'EG', 'elo': 1696, 'p_win': 0.21948608163269434, 'p_draw': 0.23387964065184141, 'p_advance': 0.45336572228453575}, {'opponent': 'CH', 'elo': 1891, 'p_win': 0.07323235933164607, 'p_draw': 0.13680607298891234, 'p_advance': 0.2100384323205584}, {'opponent': 'EN', 'elo': 2024, 'p_win': 0.03163021743873573, 'p_draw': 0.07927429661082744, 'p_advance': 0.11090451404956317}, {'opponent': 'ES', 'elo': 2157, 'p_win': 0.013295191161627695, 'p_draw': 0.04232272242262377, 'p_advance': 0.055617913584251465}], 'title_given_pass_argentina': 0.00058736946507725, 'title_from_argentina': 4.0784904142047595e-05, 'title_from_group_fixed_route': 9.527353607582318e-06, 'title_from_qualifiers_fixed_route': 1.2271231446566025e-06, 'knockout_mc': {'successes': 3, 'n': 10000, 'p': 0.0003, 'ci95': [6.187148574838712e-05, 0.0008764745225140007]}, 'total_goals_assumption': 2.7, 'n': 10000}}
def compare(a, b, path="resultado"):
    if isinstance(a, dict):
        assert a.keys() == b.keys(), path
        for key in a: compare(a[key], b[key], path + "." + key)
    elif isinstance(a, list):
        assert len(a) == len(b), path
        for i, (x,y) in enumerate(zip(a,b)): compare(x,y,path+f"[{i}]")
    elif isinstance(a, (int,float)):
        assert np.isclose(a,b,rtol=1e-9,atol=1e-12), (path,a,b)
    else: assert a == b, (path,a,b)


compare(r, REFERENCE)
assert len(list((OUT / "figures").glob("*.png"))) == 3
print("Resultados de esta parte verificados.")

## Fórmulas con datos, despejes y resultados

### Eliminatoria: estimación con sus repeticiones

$\widehat p_F=\frac{1288}{10000}=0.1288=12.88\%$

$IC_{95\%}=[0.1222933,\ 0.1355237]$

Cabo Verde terminó primero en 1288 de los 10 000 grupos simulados. Se usaron Elo 1474 para Cabo Verde y 1625 para Camerún; al recibirlo, la diferencia es 1474 − 1625 + 100 = −51. El intervalo exacto se calcula con 1288 éxitos y 10 000 ensayos, no con el número de partidos.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
print("Estimación, IC y delta local ante Camerún:",1288/10000,interval(1288,10000),1474-1625+100)

### Grupo mundialista: dos denominadores

$\widehat P(A)=\frac{2336}{10000}=0.2336=23.36\%$

$\widehat P(A\mid T)=\frac{9531}{10000}=0.9531=95.31\%$

A significa avanzar y T obtener tres empates. La primera fracción usa grupos sin imponer esos empates; la segunda, 10 000 repeticiones generadas bajo esa condición. No son 9531 clasificaciones dentro de los 2336 éxitos de la primera simulación.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
print("Avance sin condición y con tres empates:",2336/10000,9531/10000)

### Obtener los tres empates: producto real

$P(T)=0.04232272\times0.13630413\times0.25817267=1.489337\times10^{-3}$

$\widehat P(A\cap T)=(1.489337\times10^{-3})(0.9531)=1.419487\times10^{-3}$

Las probabilidades individuales corresponden a España, Uruguay y Arabia Saudita, con Elo de Cabo Verde igual a 1578 y localía neutral. El segundo cálculo combina la probabilidad de obtener los empates con la estimación condicional de avanzar después de ellos.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
ds=[probabilities(1578-r["cape_verde"]["world_elo"][code],2.7)[1] for code in ["ES","UY","SA"]]
print("Empates individuales, producto y avance conjunto:",ds,np.prod(ds),np.prod(ds)*.9531)

### Escenarios condicionados: conteos retenidos

$\widehat P(A\mid T,U)=\frac{2414}{2576}=0.937112$

$\widehat P(A\mid T,\mathrm{tercero})=\frac{8352}{8679}=0.962323$

U significa que Uruguay no pierde ante España. De las repeticiones con tres empates, 2576 cumplen U; 2414 de ellas clasifican. En el segundo escenario se retienen los 8679 terceros y 8352 avanzan. Los denominadores expresan qué condición se está evaluando.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
print("Tres empates y Uruguay no pierde:",2414/2576,interval(2414,2576))
print("Tres empates y tercer puesto:",8352/8679,interval(8352,8679))
print("Uruguay no pierde sin forzar empates:",619/2606)

### Ruta al título: cada rival aporta un factor

$p_{EG}=0.219486+0.233880=0.453366$

$p_T=0.453366\times0.210038\times0.110905\times0.055618\approx0.000587369$

Los factores corresponden a Egipto, Suiza, Inglaterra y España: victoria más empate, porque el ejercicio garantiza avanzar por penales. El producto equivale a 0.0587369 %, condicionado a haber superado a Argentina. Los valores internos no se redondean antes de multiplicar.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
factors=np.array([x["p_win"]+x["p_draw"] for x in r["cape_verde"]["knockouts"][1:]])
print("Factores, producto y porcentaje:",factors,np.prod(factors),100*np.prod(factors))

### Cambiar el punto de partida: productos

$p_{AR+}=0.0694365413(0.000587369465)=4.078490\times10^{-5}$

$p_{grupo}=0.2336\,p_{AR+}=9.527354\times10^{-6}$

Incluir Argentina agrega la probabilidad de victoria o empate contra esa selección. Empezar desde el grupo agrega el 23.36 % de avance estimado; empezar desde la eliminatoria agrega además 0.1288 y da aproximadamente 1.227123 × 10⁻⁶. Todos mantienen la ruta fija y los supuestos de independencia entre etapas.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
cv=r["cape_verde"]
p=cv["knockouts"][0]["p_advance"]*cv["title_given_pass_argentina"]
print("Desde Argentina, grupo y eliminatoria:",p,.2336*p,.1288*.2336*p)

### Derivación: condicionar un marcador a empate

$P(G_1=k,G_2=k\mid G_1=G_2)=\frac{P(G_1=k)P(G_2=k)}{\sum_{r=0}^{\infty}P(G_1=r)P(G_2=r)}$

$P(A\cap T)=P(A\mid T)P(T)\approx0.9531(0.001489337)=0.001419487$

La definición de probabilidad condicional divide la probabilidad conjunta por la del evento condicionante. La independencia de goles permite factorizar el numerador. El denominador suma todos los empates posibles: 0–0, 1–1 y los demás; por eso forzar tres empates no equivale a fijarlos todos en cero.

In [ ]:
# Recalcula la sustitución numérica con los valores de esta parte.
print("Probabilidad conjunta de tres empates y avance:",r["cape_verde"]["p_three_draws"]*r["cape_verde"]["scenarios"]["tres_empates"]["p"])

## Conclusiones

### Conclusión estadística: Cabo Verde

Se concluye que Cabo Verde tenía una posibilidad minoritaria, pero apreciable, de clasificación directa bajo el modelo: 12.88 %, con un intervalo Monte Carlo del 95 % de 12.23 % a 13.55 %. En el grupo mundialista, la estimación de avance es 23.36 % y aumenta a 95.31 % al condicionar los tres empates; esto describe situaciones distintas y no implica que conseguir esos empates sea fácil. La probabilidad de superar la ruta desde Egipto hasta España es aproximadamente 0.05874 %, incluso garantizando el avance en los empates, por lo que el título resulta muy poco probable en ese escenario. Estadísticamente, estas cifras cuantifican la dificultad de las etapas bajo fuerzas Elo, tasas de goles y rivales fijados; no constituyen una probabilidad general de ganar el Mundial. Los intervalos miden únicamente error de simulación y no incorporan incertidumbre de parámetros ni posibles errores del modelo.

## Referencias
- [World Football Elo Ratings](https://www.eloratings.net/)
- [CAF: formato clasificatorio](https://www.cafonline.com/news/everything-you-need-to-know-about-2026-world-cup-qualifying-for-africa/)
- [FIFA: clasificación y desempates](https://www.fifa.com/es/tournaments/mens/worldcup/canadamexicousa2026/articles/grupos-como-funcionan-clasificacion-criterios-desempate)
- [SciPy: Skellam](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skellam.html)
- Goldsman y Goldsman, libro del curso, capítulos 1 y 4–6.

Las URL documentan la procedencia; no se necesita acceder a ellas para ejecutar el análisis.

## Descargar tablas, datos y gráficas
El ZIP incluye los datos necesarios de esta parte, los procesados y los resultados.

In [ ]:
bundle = ROOT / "resultados_cabo_verde.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for directory in (ROOT / "data", OUT):
        for file in directory.rglob("*"):
            if file.is_file(): z.write(file, file.relative_to(ROOT))
print("Resultados de esta parte:", bundle)


# Descarga final · ambas partes
El ZIP conserva carpetas distintas para Leverkusen y Cabo Verde, incluyendo fuentes, tablas y gráficas.

In [ ]:
from pathlib import Path
import zipfile
combined = Path.cwd() / "resultados_futbol.zip"
with zipfile.ZipFile(combined, "w", zipfile.ZIP_DEFLATED) as archive:
    for case in ["leverkusen_colab", "cabo_verde_colab"]:
        for file in (Path.cwd() / case).rglob("*"):
            if file.is_file() and file.suffix != ".zip":
                archive.write(file, file.relative_to(Path.cwd()))
try:
    from google.colab import files
    files.download(str(combined))
except ImportError:
    print("Paquete completo:", combined)
